# Infosys Springboard 7.0 — Milestone 2 (Streamlit)

**One Streamlit application:** your Milestone 1 authentication portal is retained and extended with the FreightQuote AI Milestone 2 workflow. This notebook does not launch FastAPI or Uvicorn.

Run all cells in order. Add the secrets listed in the setup cell to Colab Secrets first.

In [ ]:
!pip install -q streamlit==1.66.1 streamlit-option-menu pyngrok PyJWT bcrypt pandas plotly numpy requests scikit-learn scipy matplotlib reportlab langgraph vaderSentiment pdfplumber

## 1. Mount Drive and load Colab Secrets

Required Colab Secrets: `JWT_SECRET`, `ADMIN_USERNAME`, `ADMIN_PASSWORD`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`, `NGROK_AUTHTOKEN`. `ADMIN_EMAIL_ID` is optional. Enable notebook access for each secret. Do not paste passwords, tokens or API keys into notebook cells.

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
DB_DIR = Path('/content/drive/MyDrive/Infosys_Portal')
DB_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DB_PATH'] = str(DB_DIR / 'infosys_portal.db')
os.environ['APP_KIND'] = 'Freight'
os.environ['FREIGHTQUOTE_DATA_DIR'] = str(DB_DIR / 'FreightQuoteData')

REQUIRED = ['JWT_SECRET','ADMIN_USERNAME','ADMIN_PASSWORD','EMAIL_ADDRESS','EMAIL_PASSWORD','NGROK_AUTHTOKEN']
missing=[]
for key in REQUIRED + ['ADMIN_EMAIL_ID']:
    try: value=userdata.get(key)
    except Exception: value=None
    if value: os.environ[key]=str(value)
    elif key in REQUIRED: missing.append(key)
    print(f'{key}: {"Configured ✅" if value else ("Optional" if key=="ADMIN_EMAIL_ID" else "MISSING ❌")}')
if missing: raise RuntimeError('Add these Colab Secrets and enable notebook access: '+', '.join(missing))

Path('.streamlit').mkdir(exist_ok=True)
Path('.streamlit/config.toml').write_text('[theme]\nbase="light"\nprimaryColor="#0891b2"\nbackgroundColor="#f8fafc"\nsecondaryBackgroundColor="#eef2ff"\ntextColor="#0f172a"\n[browser]\ngatherUsageStats=false\n')
print('Milestone 1 authentication database:',os.environ['DB_PATH'])
print('FreightQuote operational database folder:',os.environ['FREIGHTQUOTE_DATA_DIR'])

In [ ]:
import os
os.makedirs('/content/freight_backend', exist_ok=True)

In [ ]:
%%writefile /content/freight_backend/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


In [ ]:
%%writefile /content/freight_backend/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "Which quotes have the lowest margin?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). The matching *reorder* tickets are raised by the agent."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None):
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql)
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


In [ ]:
%%writefile /content/freight_backend/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#12a594", "#3e63dd", "#f5a524", "#e5484d", "#8e4ec6", "#30a46c", "#d6409f", "#8b8d98"
PRI = "#5b5bd6"
PAL = [PRI, TEAL, AMBER, VIOLET, RED, GREEN, PINK, BLUE, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


In [ ]:
%%writefile /content/freight_backend/config.py
"""Shared configuration. APP_KIND (Franchise | Freight) is set by the launcher. No Google Drive prompts: data lives in DATA_DIR (set PERSIST_DIR to keep it between sessions)."""
import os
KIND = os.environ.get("APP_KIND", "Franchise")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_default = "/content/app_data" if os.path.isdir("/content") else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FRANCHISEOPS_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(DATA_DIR, "franchise_database.db" if KIND == "Franchise" else "freight_database.db")
def _secret(key, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(key)
        if v: return str(v).strip()
    except Exception: pass
    return str(os.environ.get(key, default) or "").strip()
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGINGFACE_TOKEN")
def _persisted_secret():
    """No JWT_SECRET_KEY given: generate a random one once and keep it beside the database, so sessions survive restarts and are never signed with a public default."""
    import secrets as _s
    p = os.path.join(DATA_DIR, ".jwt_secret")
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = _s.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return _s.token_urlsafe(48)
JWT_SECRET_KEY = _secret("JWT_SECRET_KEY") or _persisted_secret()
ADMIN_EMAIL_ID = _secret("ADMIN_EMAIL_ID", "admin@infosys.com"); DEMO_MODE = os.environ.get("DEMO_MODE", "1") != "0"     # 1 = seeded demo accounts shown on the login page (Colab demo). 0 = production: only the admin exists, with ADMIN_PASSWORD (or a random one printed once in the server log)
def _admin_pw():
    v = _secret("ADMIN_PASSWORD")
    if v: return v
    if DEMO_MODE: return "Admin@123"
    import secrets as _s; p = os.path.join(DATA_DIR, ".admin_initial");
    if os.path.exists(p): return open(p).read().strip()
    v = _s.token_urlsafe(9) + "1aA"; open(p, "w").write(v); print("\n*** DEMO_MODE=0: initial admin password:", v, "(stored in", p, ") ***\n"); return v
ADMIN_PASSWORD = _admin_pw()
EMAIL_ID = _secret("EMAIL_ID"); EMAIL_PASSWORD = _secret("EMAIL_PASSWORD").replace(" ", "")
MODEL_NAME = os.environ.get("LLM_MODEL", "")
NLLB_MODEL = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-600M")


In [ ]:
%%writefile /content/freight_backend/db.py
import sqlite3, os, threading
import pandas as pd
from config import DB_PATH

# A single, long-lived connection reused across the whole app process
# instead of opening (and never explicitly closing) a brand-new SQLite
# connection + re-running PRAGMA setup on every single call. That old
# pattern was leaking file handles under load and was the single
# biggest source of slowness/lock contention across the app.
_conn = None
_conn_lock = threading.Lock()

def get_conn():
    global _conn
    if _conn is None:
        with _conn_lock:
            if _conn is None:
                os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
                _conn = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False)
                _conn.execute("PRAGMA journal_mode=WAL;")
                _conn.execute("PRAGMA synchronous=NORMAL;")
                _conn.execute("PRAGMA cache_size=-64000;")   # ~64MB page cache
                _conn.execute("PRAGMA temp_store=MEMORY;")
    return _conn

def save_chat_message(username, role, message):
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception: pass

def load_chat_history(username=None, limit=100):
    try:
        with get_conn() as conn:
            if username:
                df = pd.read_sql("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", conn, params=(username, limit))
                if df.empty:
                    df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))
            else:
                df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))

            res = []
            for _, r in df.iterrows():
                content_val = str(r.get("message") or r.get("content") or "")
                res.append({
                    "role": str(r.get("role", "assistant")),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception: return []

def clear_chat_history(username=None):
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception: pass

def init_db():
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        # Milestone 2 authentication columns — safe migrations for existing databases.
        for _col, _ddl in [
            ("username", "ALTER TABLE users ADD COLUMN username TEXT"),
            ("security_question", "ALTER TABLE users ADD COLUMN security_question TEXT"),
            ("security_answer_hash", "ALTER TABLE users ADD COLUMN security_answer_hash TEXT"),
            ("failed_attempts", "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0"),
            ("lock_until", "ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL"),
            ("account_status", "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'"),
            ("profile_picture", "ALTER TABLE users ADD COLUMN profile_picture TEXT DEFAULT NULL"),
        ]:
            try:
                conn.execute(_ddl)
            except Exception:
                pass

        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.commit()


In [ ]:
%%writefile /content/freight_backend/domain_freight.py
"""FreightQuote AI live domain (PRS v3): 7 agents, ML pricing + congestion models, vessel tracking, reroute loop, PDF documents."""
import sys, os, re, json, math, random, time, datetime, zlib
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit, run_cycle, _log_run, act, init_live_tables
from config import DB_PATH
KIND = "Freight"; NAME = "FreightQuote AI"; ENTITY_COL = "shipment_id"; SMS = False; ESCALATE_TO = "Operations Manager"
OWNER = {"quote_anomaly": "Finance Lead", "reroute": "Operations Manager", "quote_approval": "Admin", "customs_docs": "Customs Officer", "congestion_notice": "Operations Manager", "margin_review": "Finance Lead", "quote_hold": "Freight Broker"}
ACTION_AGENT = {"quote_anomaly": "anomaly_watch", "reroute": "weather_watch", "customs_docs": "customs_watch", "congestion_notice": "route_watch", "quote_approval": "quote_watch", "margin_review": "quote_watch", "quote_hold": "quote_watch"}
DATA_DIR = os.path.dirname(DB_PATH); TICK_NM = 84.0       # 1 tick = 1 sailing hour
CARGO = {"Electronics": 1.15, "Pharmaceuticals": 1.35, "Automotive Parts": 1.1, "Textiles": .95, "Heavy Machinery": 1.3, "Perishables": 1.4, "General": 1.0, "Hazardous": 1.6, "Bulk": .8}
CODES = {k: i for i, k in enumerate(CARGO)}
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- geo + routing
def nm(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 3440.065 * 2 * math.asin(min(1, math.sqrt(h)))
def chain_nm(ch): return sum(nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1))
def interp(ch, frac):
    legs = [nm(ch[i], ch[i + 1]) for i in range(len(ch) - 1)]; tot = sum(legs) or 1; d = min(max(frac, 0), 1) * tot
    for i, L in enumerate(legs):
        if d <= L or i == len(legs) - 1:
            t = d / L if L else 0; return (ch[i][0] + (ch[i + 1][0] - ch[i][0]) * t, ch[i][1] + (ch[i + 1][1] - ch[i][1]) * t)
        d -= L
SUEZ_S, SUEZ_N, CAPE = (12.6, 43.4), (30.0, 32.55), (-34.4, 18.5)
_ACC = {}
def _access(xy):
    """Sea entry point for river / bay ports (Hamburg, Wuhan, Houston ...), looked up by the port's coordinates."""
    if not _ACC:
        try:
            import seanav
            for n, la, lo in Q("SELECT port_name, lat, lon FROM ports").itertuples(index=False): _ACC[(round(float(la), 2), round(float(lo), 2))] = seanav.access_for(n)
        except Exception: _ACC[None] = None
    return _ACC.get((round(float(xy[0]), 2), round(float(xy[1]), 2)))
def best_route(o, d):
    import seanav
    r = seanav.sea().route((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), (), (), _access(o), _access(d))
    return {"name": "via " + " + ".join(r["via"]) if r["via"] else "direct sea route", "chain": r["pts"], "nm": round(r["nm"]), "via": r["via"]}
def route_options(o, d, avoid=()):
    """Real sea routes (water only: see seanav.py). avoid may contain 'suez', 'panama', 'malacca', 'hormuz', 'bosphorus' and/or 'storm' (steer around the active storms)."""
    import seanav
    S = seanav.sea(); blocked = tuple(a for a in avoid if a in seanav.BLOCK_GROUPS); st = tuple((s["lat"], s["lon"], s.get("r", 700)) for s in storms()) if "storm" in avoid else ()
    opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, st, _access(o), _access(d))
    if not opts and st: opts = S.options((float(o[0]), float(o[1])), (float(d[0]), float(d[1])), blocked, (), _access(o), _access(d))
    return [{"name": x["name"], "chain": x["pts"], "nm": round(x["nm"]), "via": x["via"]} for x in opts]
def port_xy(name):
    r = Q("SELECT lat, lon, congestion_index ci, avg_dwell_days dw, country FROM ports WHERE port_name=? LIMIT 1", (name,))
    return (float(r.lat[0]), float(r.lon[0]), float(r.ci[0]), float(r.dw[0]), r.country[0]) if len(r) else None

# ---------------------------------------------------------------- weather (Open-Meteo live + cached port rows + simulated storms)
_WX = {}; _OFF = [0.0]
def storms(): return get_state("storms", []) or []
def inject_storm(lat=None, lon=None, wind=22.0):
    s = storms(); s.append({"lat": lat if lat is not None else random.uniform(-30, 45), "lon": lon if lon is not None else random.uniform(-60, 120), "r": 700, "wind": wind, "ts": now()}); set_state("storms", s[-6:])
def weather_at(lat, lon, allow_live=True):
    k = (round(lat / 2) * 2, round(lon / 2) * 2); hit = _WX.get(k); wind, wave, src = 6.0, 1.2, "cached"
    if hit and time.time() - hit[0] < 900: wind, wave, src = hit[1:]
    elif allow_live and time.time() > _OFF[0]:
        try:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=wind_speed_10m&wind_speed_unit=ms", timeout=3).json()["current"]
            wind, src = float(r["wind_speed_10m"]), "Open-Meteo live"
            try: wave = float(requests.get(f"https://marine-api.open-meteo.com/v1/marine?latitude={lat}&longitude={lon}&current=wave_height", timeout=3).json()["current"]["wave_height"] or 1.2)
            except Exception: wave = 1.2
            _WX[k] = (time.time(), wind, wave, src)
        except Exception: _OFF[0] = time.time() + 120; wind, src = 5 + 4 * random.random(), "simulated (offline)"
    else: wind, src = 5 + 4 * random.random(), "simulated (offline)"
    for s in storms():
        d = nm((lat, lon), (s["lat"], s["lon"]))
        if d < s["r"]: wind = max(wind, s["wind"] * (1 - .5 * d / s["r"])); src += " + storm"
    return {"wind": round(wind, 1), "wave": round(wave, 1), "src": src, "level": "High" if wind > 15 else "Medium" if wind >= 8 else "Low"}
def wscore(level): return {"High": 1.0, "Medium": .5, "Low": .12}[level]

# ---------------------------------------------------------------- ML models
def _gen_rates(n=4000):
    R = np.random.default_rng(42); dist = R.uniform(500, 12000, n); w = R.uniform(1, 600, n); cg = R.integers(0, len(CARGO), n); cong = R.integers(0, 4, n); mult = np.array(list(CARGO.values()))
    cost = (250 + .21 * dist) * (.6 + .012 * w) * (1 + .08 * cong) * mult[cg] * R.normal(1, .06, n)
    return pd.DataFrame({"distance_nm": dist, "cargo_weight": w, "cargo_type": cg, "congestion_level": cong, "freight_cost_usd": cost})
_ML = {}
def pricing_model():
    from sklearn.ensemble import RandomForestRegressor; from sklearn.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error; import joblib
    path = os.path.join(DATA_DIR, "pricing_rf.joblib")
    if "rf" in _ML: return _ML
    if os.path.exists(path): _ML.update(joblib.load(path)); return _ML
    src = "bootstrap synthetic (feed a Kaggle freight-rate CSV to replace)"
    try:
        d = Q("SELECT * FROM freight_rates")
        if len(d) >= 300: src = "fed dataset (freight_rates table)"
        else: d = _gen_rates()
    except Exception: d = _gen_rates()
    X, y = d[["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]], d.freight_cost_usd
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42); rf = RandomForestRegressor(100, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    m = {"r2": round(r2_score(yte, rf.predict(Xte)), 3), "rmse": round(float(np.sqrt(mean_squared_error(yte, rf.predict(Xte)))), 1), "source": src, "importance": dict(zip(X.columns, rf.feature_importances_.round(3).tolist()))}
    _ML.update(rf=rf, metrics=m); joblib.dump({"rf": rf, "metrics": m}, path)
    with _DB_LOCK, conn() as c: c.execute("UPDATE ml_models SET is_active=0 WHERE model_name='rf_freight_price'"); c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,1)", ("rf_freight_price", json.dumps(m), now()))
    return _ML
def congestion_model():
    from sklearn.ensemble import RandomForestClassifier
    if "cg" in _ML: return _ML["cg"]
    R = np.random.default_rng(3); n = 3000; ci = R.uniform(1, 4.8, n); dw = R.uniform(1.5, 7, n); mo = R.integers(1, 13, n); dow = R.integers(0, 7, n); cg = R.integers(0, len(CARGO), n)
    lvl = np.clip(np.round((ci - 1) / 1.3 + .25 * (dw > 5) + .35 * np.isin(mo, [10, 11, 12, 1, 2]) + R.normal(0, .3, n)), 0, 3).astype(int)
    clf = RandomForestClassifier(100, random_state=42).fit(np.c_[ci, dw, mo, dow, cg], lvl); _ML["cg"] = clf; return clf
def predict_congestion(ci, dw, cargo="General"):
    t = datetime.date.today(); return int(congestion_model().predict([[ci, dw, t.month, t.weekday(), CODES.get(cargo, 6)]])[0])

# ---------------------------------------------------------------- bootstrap, simulator, vessel tracking
def bootstrap():
    from scale_data import scale_freight; scale_freight()
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS quote_requests(req_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, origin TEXT, dest TEXT, cargo TEXT, weight_t REAL, hs_code TEXT, customer_id TEXT, language TEXT, status TEXT DEFAULT 'new', quote_id TEXT, result TEXT, docs_provided TEXT);
        CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc));
        CREATE TABLE IF NOT EXISTS vessels(shipment_id TEXT PRIMARY KEY, lat REAL, lon REAL, progress REAL, chain TEXT, route_name TEXT, speed_kn REAL, updated TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE TABLE IF NOT EXISTS customs_rules(chapter TEXT PRIMARY KEY, goods TEXT, duty_pct REAL, docs TEXT, restricted INT);""")
        try: c.execute("ALTER TABLE quote_requests ADD COLUMN docs_provided TEXT")
        except Exception: pass
        if not c.execute("SELECT 1 FROM customs_rules").fetchone():
            base = "Commercial Invoice;Packing List;Bill of Lading"
            for ch, g, d, ex, rs in [("30", "Pharmaceuticals", 5, "Import License;Drug Control Certificate", 1), ("84", "Machinery", 7.5, "Certificate of Origin", 0), ("85", "Electronics", 10, "BIS Certificate;Certificate of Origin", 0), ("87", "Vehicles and parts", 15, "Homologation Certificate", 0),
                                     ("10", "Cereals", 12, "Phytosanitary Certificate", 0), ("62", "Apparel", 12, "Certificate of Origin", 0), ("27", "Mineral fuels", 5, "Hazmat Declaration;IMDG Certificate", 1), ("39", "Plastics", 8, "Certificate of Origin", 0), ("72", "Iron and steel", 7.5, "Mill Test Certificate", 0), ("22", "Beverages", 20, "Excise Permit", 1)]:
                c.execute("INSERT INTO customs_rules VALUES(?,?,?,?,?)", (ch, g, d, base + ";" + ex, rs))
        n = c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]
        if n == 0:
            R = random.Random(5)
            for sid, o, d in c.execute("SELECT shipment_id, origin_port, dest_port FROM shipments WHERE status IN ('In Transit','Port Congestion Delay','Anchorage Pending')").fetchall():
                po, pd_ = port_xy(o), port_xy(d)
                if not po or not pd_: continue
                opt = best_route(po[:2], pd_[:2]); p = R.uniform(.05, .9); la, lo = interp(opt["chain"], p)
                c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, la, lo, p, json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
    pricing_model(); congestion_model()

def tick():
    reasons = []; R = random
    with _DB_LOCK, conn() as c:
        real = set()
        for sid, p, chain, spd in c.execute("SELECT shipment_id, progress, chain, speed_kn FROM vessels").fetchall():
            if sid in real: continue
            ch = json.loads(chain); tot = chain_nm(ch) or 1; p2 = p + spd * 1.0 / tot
            if p2 >= 1:
                c.execute("DELETE FROM vessels WHERE shipment_id=?", (sid,)); c.execute("UPDATE shipments SET status='Delivered' WHERE shipment_id=?", (sid,)); emit("delivered", sid); continue
            la, lo = interp(ch, p2); c.execute("UPDATE vessels SET lat=?, lon=?, progress=?, updated=? WHERE shipment_id=?", (la, lo, p2, now(), sid))
        _tickn = get_state("tickn", 0) + 1; set_state("tickn", _tickn)
        ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
        try: import weather as _WXL; _fresh = _WXL.fresh_ports()
        except Exception: _fresh = set()
        for i in range(12):                                         # round-robin refresh of port weather + congestion drift
            nme, la, lo = ports[(_tickn * 12 + i) % len(ports)]; w = weather_at(la, lo, allow_live=False)
            if nme in _fresh:                                       # real Open-Meteo reading is recent: keep it, only drift congestion
                c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme)); continue
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (nme, w["level"], w["src"], w["wind"] * 3.6, w["wave"], 24.0))
            c.execute("UPDATE ports SET congestion_index=MIN(5,MAX(0.5,congestion_index+?)) WHERE port_name=?", (R.uniform(-.12, .12), nme))
        for _ in range(max(0, min(4, 230 - c.execute("SELECT COUNT(*) FROM vessels").fetchone()[0]))):   # keep the tracked fleet topped up as ships arrive
            o, d = R.sample(ports, 2); sid = f"SHP-L{R.randint(0, 10**8):08d}"; opt = best_route((o[1], o[2]), (d[1], d[2]))
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, o[0], d[0], R.choice([r[0] for r in c.execute("SELECT name FROM carriers").fetchall()]), "In Transit", (datetime.date.today() + datetime.timedelta(days=20)).isoformat(), R.uniform(2000, 25000), f"HS-{R.choice([30,84,85,87,10,62])}{R.randint(10,99)}", .1))
            c.execute("INSERT OR REPLACE INTO vessels VALUES(?,?,?,?,?,?,?,?)", (sid, o[1], o[2], R.uniform(0, .05), json.dumps(opt["chain"]), opt["name"], R.uniform(12, 17), now()))
        if _tickn % 6 == 0 and R.random() < .6:                      # incoming customer quote request
            o, d = R.sample(ports, 2); cu = c.execute("SELECT customer_id FROM customers ORDER BY RANDOM() LIMIT 1").fetchone()[0]
            hs = f"HS-{R.choice([30,84,85,87,10,62,27])}{R.randint(10,99)}"; req_docs = _required_docs(hs)[1]; given = [x for x in req_docs if R.random() < (.97 if x in ("Commercial Invoice", "Packing List", "Bill of Lading") else .82)]
            c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), o[0], d[0], R.choice(list(CARGO)), round(R.uniform(5, 450), 1), hs, cu, "English", ";".join(given))); reasons.append("quote-request")
        if _tickn % 40 == 0 and R.random() < .5: inject_storm(); reasons.append("storm")
        if _tickn % 25 == 0: set_state("storms", [s for s in storms() if (time.time() % 1e9) > 0][-3:])
    return reasons

def feed():
    with _DB_LOCK, conn() as c: ports = c.execute("SELECT port_name, lat, lon FROM ports ORDER BY port_id").fetchall()
    k = get_state("feedk", 0) or 0
    for i in range(10): _, la, lo = ports[(k * 10 + i) % len(ports)]; weather_at(la, lo, allow_live=True)
    set_state("feedk", k + 1)
DEFAULT_POLICY = {"reroute": "approval"}
def process():
    for r in Q("SELECT req_id FROM quote_requests WHERE status='new' ORDER BY req_id LIMIT 3").req_id.tolist(): run_quote(int(r))

# ---------------------------------------------------------------- the 7 pipeline agents
def route_agent(s):
    req = s["req"]; po, pd_ = port_xy(req["origin"]), port_xy(req["dest"])
    if not po or not pd_: raise ValueError("unknown port")
    opts = route_options(po[:2], pd_[:2], s.get("avoid", ())); best = opts[0]
    cong = predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, req["cargo"]); days = best["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
    cars = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); cars["score"] = cars.rating / 5 * 40 + cars.on_time_pct * .4 - cars.avg_cost_index * 12 - (cars.risk_level == "High") * 8
    top = cars.sort_values("score", ascending=False).head(3)
    return AgentResult("route", 100 - cong * 20, risk_label(100 - cong * 20), f"{best['name']}, {best['nm']:,} nm, {days:.1f} days, congestion level {cong}/3 (ML)",
        {"route": best["name"], "chain": best["chain"], "distance_nm": best["nm"], "transit_days": round(days, 1), "congestion": cong, "alternatives": [o["name"] for o in opts[1:]], "carriers": top.name.tolist(), "ports": [po[:2], pd_[:2]]})
def weather_agent(s):
    r = s["results"]["route"].findings; ch = r["chain"]; pts = [interp(ch, f) for f in (0, .25, .5, .75, 1)]; ws = [weather_at(la, lo) for la, lo in pts]; worst = max(ws, key=lambda w: w["wind"])
    sc = max(wscore(w["level"]) for w in ws); delay = round(min(.95, sc * .6 + worst["wind"] / 60), 2)
    return AgentResult("weather", 100 - sc * 100, risk_label(100 - sc * 100), f"{worst['level']} risk: wind {worst['wind']} m/s, waves {worst['wave']} m ({worst['src']}), delay probability {delay}", {"level": worst["level"], "wind": worst["wind"], "wave": worst["wave"], "risk": sc, "delay_prob": delay, "src": worst["src"]})
def pricing_agent(s):
    m = pricing_model(); rt = s["results"]["route"].findings; req = s["req"]
    ml = float(m["rf"].predict(pd.DataFrame([[rt["distance_nm"], req["weight_t"], CODES.get(req["cargo"], 6), rt["congestion"]]], columns=["distance_nm", "cargo_weight", "cargo_type", "congestion_level"]))[0])
    fuel = ml * float(get_state("fuel_pct", .12) or .12); fees = 350 + 40 * rt["congestion"] * 2
    return AgentResult("pricing", 90, "Low", f"ML base ${ml:,.0f} (R2 {m['metrics']['r2']}), fuel 12% ${fuel:,.0f}, port fees ${fees:,.0f}", {"base": round(ml, 2), "fuel": round(fuel, 2), "port_fees": fees, "model": m["metrics"]})
def _required_docs(hs):
    ch = (re.findall(r"\d+", str(hs)) or ["00"])[0][:2]; r = Q("SELECT * FROM customs_rules WHERE chapter=?", (ch,))
    base = ["Commercial Invoice", "Packing List", "Bill of Lading"]
    return ch, (r.docs[0].split(";") if len(r) else base), (int(r.restricted[0]) if len(r) else 0), (float(r.duty_pct[0]) if len(r) else 10.0), (r.goods[0] if len(r) else "General cargo")
def _attached(ref):
    try: return set(Q("SELECT doc FROM shipment_docs WHERE ref=?", (ref,)).doc)
    except Exception: return set()
def customs_agent(s):
    """Customs check against the customs_rules table: duty, restricted goods, and which required documents the customer actually provided or attached later."""
    req = s["req"]; ch, docs, restricted, duty, goods = _required_docs(req["hs_code"]); prov = req.get("docs_provided")
    have = set(str(prov).split(";")) if isinstance(prov, str) and prov else set(docs)                # legacy requests without a document list count as complete
    have |= _attached(s.get("quote_id", "")); missing = [d for d in docs if d not in have]
    risk = round(min(1, .12 + .35 * restricted + .18 * len(missing)), 2)
    return AgentResult("customs", 100 - risk * 100, risk_label(100 - risk * 100), f"Chapter {ch} ({goods}): {'restricted, ' if restricted else ''}{len(missing)} document(s) missing, duty {duty}%", {"risk": risk, "docs": docs, "missing": missing, "duty_pct": duty, "restricted": bool(restricted), "status": "Hold" if len(missing) or risk > .6 else "Ready", "goods": goods})
def margin_agent(s):
    p = s["results"]["pricing"].findings; cust = Q("SELECT priority_tier t, credit_risk cr FROM customers WHERE customer_id=?", (s["req"]["customer_id"],)); tier = cust.t[0] if len(cust) else "Silver"; cr = float(cust.cr[0]) if len(cust) else .1
    target = .20 + {"Platinum": -.02, "Gold": 0, "Silver": .02, "Bronze": .04}.get(tier, .02) + (.02 if cr > .3 else 0)
    cost = p["base"] + p["fuel"] + p["port_fees"]; op = p["base"] * .15; duty = p["base"] * s["results"]["customs"].findings["duty_pct"] / 100 * .1; total_cost = cost + op + duty
    price = total_cost / (1 - target); margin = (price - total_cost) / price * 100
    return AgentResult("margin", min(100, margin * 4), "Low" if margin >= 10 else "High", f"Target {target*100:.0f}% ({tier}), price ${price:,.0f}, margin {margin:.1f}%", {"price": round(price, 2), "margin_pct": round(margin, 1), "op_cost": round(op, 2), "target": target, "tier": tier, "cost": round(total_cost, 2)})
def document_agent(s):
    q = s["quote_id"]; out = os.path.join(DATA_DIR, "docs"); os.makedirs(out, exist_ok=True); req = s["req"]; rt = s["results"]["route"].findings; paths = []
    try:
        from reportlab.lib.pagesizes import A4; from reportlab.pdfgen import canvas
        for kind, lines in (("BillOfLading", [f"Shipper/Customer: {req['customer_id']}", f"Port of loading: {req['origin']}", f"Port of discharge: {req['dest']}", f"Cargo: {req['cargo']} ({req['hs_code']}), {req['weight_t']} t", f"Carrier: {rt['carriers'][0]}", f"Route: {rt['route']}, {rt['distance_nm']:,} nm"]),
                            ("PackingList", [f"Quote: {q}", f"Cargo type: {req['cargo']}", f"Gross weight: {req['weight_t']} t", f"Documents: {', '.join(s['results']['customs'].findings['docs'])}"])):
            p = os.path.join(out, f"{q}_{kind}.pdf"); cv = canvas.Canvas(p, pagesize=A4); cv.setFont("Helvetica-Bold", 16); cv.drawString(50, 800, f"{kind} - {q}"); cv.setFont("Helvetica", 11)
            for i, ln in enumerate(lines): cv.drawString(50, 770 - 22 * i, ln)
            cv.save(); paths.append(p)
    except Exception as e: return AgentResult("document", 60, "Medium", f"PDF generation unavailable: {e}")
    return AgentResult("document", 100, "Low", f"{len(paths)} PDFs generated", {"paths": paths})
def alert_agent(s):
    val = s["results"]["margin"].findings["price"]; w = s["results"]["weather"].findings; acts = []
    if val > 50000: acts.append({"type": "quote_approval", "target": s["quote_id"], "detail": f"Quote ${val:,.0f} exceeds $50,000: admin approval required", "severity": "Critical", "owner": "Admin"})
    elif s["results"]["margin"].findings["margin_pct"] < 10: acts.append({"type": "margin_review", "target": s["quote_id"], "detail": "Margin below 10%", "severity": "Warning"})
    if w["level"] == "High": acts.append({"type": "reroute", "target": s["quote_id"], "detail": f"High weather risk on lane (wind {w['wind']} m/s)", "severity": "Warning"})
    return AgentResult("alert", 100, "Low", f"{len(acts)} alerts raised", {}, acts)

def _compute(req, qid, cid=None, avoid=()):
    """The quote pipeline as a LangGraph graph: route -> weather -> [High risk: re-plan avoiding chokepoints, retry] -> pricing -> customs -> margin.
    cid=None runs it without logging (used by the Digital Twin and the quote preview)."""
    st = {"req": req, "results": {}, "avoid": tuple(avoid), "quote_id": qid, "tries": 0}
    def step(name, fn):
        t = time.time()
        try: res = fn(st)
        except Exception as e: res = AgentResult(name, 40, "High", f"error: {e}")
        st["results"][name] = res
        if cid: _log_run(cid, "quote", res, int((time.time() - t) * 1000))
        return res
    def replan():
        st["avoid"] = ("suez",); st["tries"] += 1
        if cid: _log_run(cid, "quote", AgentResult("re-plan", 100, "Low", f"weather High: re-routing avoiding Suez/Red Sea (try {st['tries']})"), 0)
    def need_replan(): return st["results"]["weather"].findings.get("level") == "High" and st["tries"] < 1 and st["results"]["route"].findings.get("alternatives") and "suez" not in st["avoid"]
    try:
        from langgraph.graph import StateGraph, END
        from typing import TypedDict
        class G(TypedDict, total=False): n: int
        gr = StateGraph(G)
        gr.add_node("route", lambda g: (step("route", route_agent), {})[1]); gr.add_node("weather", lambda g: (step("weather", weather_agent), {})[1]); gr.add_node("replan", lambda g: (replan(), {})[1])
        gr.add_node("rest", lambda g: ([step(n, f) for n, f in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent))], {})[1])
        gr.set_entry_point("route"); gr.add_edge("route", "weather"); gr.add_edge("replan", "route"); gr.add_conditional_edges("weather", lambda g: "replan" if need_replan() else "rest"); gr.add_edge("rest", END); gr.compile().invoke({"n": 0})
    except ImportError:
        step("route", route_agent); step("weather", weather_agent)
        if need_replan(): replan(); step("route", route_agent); step("weather", weather_agent)
        for nme, fn in (("pricing", pricing_agent), ("customs", customs_agent), ("margin", margin_agent)): step(nme, fn)
    return st, step

def _decide(st):
    res = st["results"]; req = st["req"]; wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    risk = round(wr * .4 + cr * .3 + rt * .3, 3); m = res["margin"].findings; val = m["price"]          # PRS v3: weather x0.40 + customs x0.30 + congestion x0.30
    decision = "On Hold" if res["customs"].findings["status"] == "Hold" or risk > .7 else "Pending Approval" if val > 50000 or m["margin_pct"] < 10 else "Approved"
    narr = f"Quote {st['quote_id']}: {req['origin']} to {req['dest']}, {res['route'].findings['route']}, {res['route'].findings['transit_days']} days. Risk score {risk} (weather {wr}, customs {cr}, congestion {rt:.2f}). Price ${val:,.0f} at {m['margin_pct']}% margin. Decision: {decision}."
    return risk, decision, val, m, narr

def preview_quote(req, avoid=()):
    """Price a lane without saving anything. The Digital Twin calls this on its database copy."""
    st, _ = _compute(dict(req), "PREVIEW", None, avoid)
    if not all(k in st["results"] and st["results"][k].findings for k in ("route", "weather", "pricing", "customs", "margin")): return None
    risk, decision, val, m, narr = _decide(st); r = st["results"]
    return {"price": val, "margin_pct": m["margin_pct"], "risk": risk, "decision": decision, "route": r["route"].findings["route"], "days": r["route"].findings["transit_days"], "weather": r["weather"].findings["level"], "narrative": narr}

def run_quote(req_id):
    from live_engine import _log_run, init_live_tables, act
    import sys; dom = sys.modules[__name__]; init_live_tables()
    r = Q("SELECT * FROM quote_requests WHERE req_id=?", (req_id,)); req = r.iloc[0].to_dict(); cid = f"Q{req_id}-" + datetime.datetime.now().strftime("%H%M%S"); qid = f"QTE-L{req_id:05d}"
    st, step = _compute(req, qid, cid); res = st["results"]
    if not all(k in res and res[k].findings for k in ("route", "weather", "pricing", "customs", "margin")):
        with _DB_LOCK, conn() as c: c.execute("UPDATE quote_requests SET status='error', result=? WHERE req_id=?", (json.dumps({k: v.summary for k, v in res.items()}), req_id)); return None
    risk, decision, val, m, narr = _decide(st); step("document", document_agent); step("alert", alert_agent); wr, cr, rt = res["weather"].findings["risk"], res["customs"].findings["risk"], res["route"].findings["congestion"] / 3
    with _DB_LOCK, conn() as c:
        sid = f"SHP-{qid[4:]}"
        c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)", (sid, req["origin"], req["dest"], res["route"].findings["carriers"][0], "Customs Hold" if decision == "On Hold" else "Quoted", (datetime.date.today() + datetime.timedelta(days=int(res["route"].findings["transit_days"]) + 3)).isoformat(), req["weight_t"] * 1000, req["hs_code"], risk))
        p = res["pricing"].findings; c.execute("INSERT OR REPLACE INTO freight_quotes VALUES(?,?,?,?,?,?,?,?,?,?,?,?)", (qid, sid, req["customer_id"], res["route"].findings["carriers"][0], p["base"], round(p["base"] * .02, 2), round(p["base"] * res["customs"].findings["duty_pct"] / 100 * .1, 2), p["fuel"], val, m["margin_pct"], decision, now()[:10]))
        c.execute("UPDATE quote_requests SET status='done', quote_id=?, result=? WHERE req_id=?", (qid, json.dumps({"decision": decision, "risk": risk, "price": val, "narrative": narr, "missing": res["customs"].findings["missing"], "route": res["route"].findings["route"], "days": res["route"].findings["transit_days"], "weather": res["weather"].findings["level"], "margin_pct": m["margin_pct"], "parts": {"weather": wr, "customs": cr, "congestion": round(rt, 2)}}), req_id))
    dec = {"score": round((1 - risk) * 100, 1), "risk": risk_label((1 - risk) * 100), "narrative": narr, "actions": res["alert"].actions + ([{"type": "quote_hold", "target": qid, "detail": "Customs hold, missing: " + ", ".join(res["customs"].findings["missing"]), "severity": "Warning"}] if decision == "On Hold" and res["customs"].findings["missing"] else [])}
    act(dom, dec, cid, {}); return dec

def attach_doc(ref, doc, by="user"):
    """Attach a missing customs document to a quote (QTE-...) or a shipment (SHP-...). Re-runs the quote so the hold lifts when nothing is missing."""
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS shipment_docs(ref TEXT, doc TEXT, ts TEXT, by TEXT, UNIQUE(ref, doc))"); c.execute("INSERT OR IGNORE INTO shipment_docs VALUES(?,?,?,?)", (ref, doc, now(), by))
    emit("doc_attached", ref, {"doc": doc})
    if ref.startswith("QTE-L"):
        try: run_quote(int(ref[5:]))
        except Exception: pass

# ---------------------------------------------------------------- fleet monitoring cycle (the same agents, run continuously)
def route_watch(s):
    p = Q("SELECT port_name, congestion_index ci, avg_dwell_days dw FROM ports WHERE congestion_index>=4").sort_values("ci", ascending=False)
    n = Q("SELECT COUNT(*) n FROM shipments WHERE status='Port Congestion Delay'").n[0]; sc = 100 - min(60, len(p) * 3 + n * .5)
    return AgentResult("route_watch", sc, risk_label(sc), f"{len(p)} ports with congestion index >= 4, {n} shipments delayed", {"ports": p.head(8).port_name.tolist(), "n_congested": len(p), "delayed": int(n)}, [{"type": "congestion_notice", "target": r.port_name, "detail": f"Congestion index {r.ci:.1f}, dwell {r.dw:.1f} days", "severity": "Warning"} for r in p.head(6).itertuples()])
def weather_watch(s):
    v = Q("SELECT shipment_id, lat, lon, chain FROM vessels"); bad = []; acts = []
    for r in v.itertuples():
        w = weather_at(r.lat, r.lon, allow_live=False)
        if w["level"] == "High": bad.append(r.shipment_id)
    for sid in bad[:15]:
        acts.append({"type": "reroute", "target": sid, "detail": "High weather at current vessel position: alternate route proposed", "severity": "Critical"})
    sc = 100 - min(90, len(bad) / max(len(v), 1) * 400); return AgentResult("weather_watch", sc, risk_label(sc), f"{len(bad)} of {len(v)} vessels in High-risk weather ({len(storms())} active storms)", {"vessels_high": bad}, acts)
def customs_watch(s):
    """Shipments on customs hold: required documents (customs_rules by HS chapter) minus documents attached. Clears the hold when nothing is missing."""
    h = Q("SELECT shipment_id, hs_code FROM shipments WHERE status='Customs Hold'"); acts = []; cleared = 0
    for r in h.itertuples():
        ch, docs, restricted, duty, goods = _required_docs(r.hs_code); miss = [d for d in docs if d not in _attached(r.shipment_id)]
        if not miss and not restricted:
            with _DB_LOCK, conn() as c: c.execute("UPDATE shipments SET status='In Transit' WHERE shipment_id=?", (r.shipment_id,)); emit("customs_cleared", r.shipment_id); cleared += 1; continue
        acts.append({"type": "customs_docs", "target": r.shipment_id, "detail": f"Customs hold ({r.hs_code}, {goods}): missing {', '.join(miss[:4]) if miss else 'restricted goods clearance'}", "severity": "Warning"})
    n = len(h) - cleared; sc = 100 - min(70, n * 2)
    return AgentResult("customs_watch", sc, risk_label(sc), f"{n} shipments on customs hold, {cleared} cleared this cycle", {"holds": n, "cleared": cleared}, acts[:25])
def quote_watch(s):
    p = Q("SELECT q.quote_id, q.final_price, q.margin_pct, q.status, r.result FROM freight_quotes q LEFT JOIN quote_requests r ON r.quote_id=q.quote_id WHERE q.status IN ('Pending Approval','On Hold')"); acts = []
    for r in p.itertuples():
        if r.status == "Pending Approval": acts.append({"type": "quote_approval", "target": r.quote_id, "detail": f"Quote ${r.final_price:,.0f} exceeds $50,000: admin approval required" if r.final_price > 50000 else "Margin below 10%: approval required", "severity": "Critical" if r.final_price > 50000 else "Warning", "owner": "Admin"})
        else:
            try: miss = json.loads(r.result or "{}").get("missing", [])
            except Exception: miss = []
            acts.append({"type": "quote_hold", "target": r.quote_id, "detail": "Customs hold, missing: " + ", ".join(miss) if miss else "On hold: shipment risk score above 0.7", "severity": "Warning"})
    sc = 100 - min(60, len(p) * 3); return AgentResult("quote_watch", sc, risk_label(sc), f"{len(p)} quotes awaiting approval or on hold", {"pending": int((p.status == 'Pending Approval').sum()), "on_hold": int((p.status == 'On Hold').sum())}, acts[:30])

def anomaly_watch(s):
    """IsolationForest over quote economics (price/base ratio, margin, fuel share) to catch mispriced or tampered quotes."""
    from sklearn.ensemble import IsolationForest
    q = Q("SELECT quote_id, base_cost, fuel_surcharge, final_price, margin_pct FROM freight_quotes WHERE base_cost>0")
    if len(q) < 30: return AgentResult("anomaly_watch", 100, "Low", "waiting for more quotes", {"flagged": []})
    X = pd.DataFrame({"ratio": q.final_price / q.base_cost, "margin": q.margin_pct, "fuel": q.fuel_surcharge / q.base_cost}); iso = IsolationForest(contamination=.06, random_state=42).fit(X); q["flag"] = iso.predict(X) == -1
    bad = q[q.flag]; sc = 100 - len(bad) / len(q) * 250
    return AgentResult("anomaly_watch", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} of {len(q)} quotes look anomalous (IsolationForest)", {"flagged": bad.quote_id.tolist()},
        [{"type": "quote_anomaly", "target": r.quote_id, "detail": f"Price/base ratio {r.final_price / r.base_cost:.2f}, margin {r.margin_pct:.1f}%: review before sending", "severity": "Warning"} for r in bad.head(10).itertuples()])

AGENTS = {"route_watch": route_watch, "weather_watch": weather_watch, "customs_watch": customs_watch, "quote_watch": quote_watch, "anomaly_watch": anomaly_watch}
REPLAN = {"weather_watch": lambda r: ["route_watch"] if r.risk != "Low" else []}
def plan(req): return list(AGENTS)
def engine(s):
    r = s["results"]; score = float(np.mean([v.score for v in r.values()])); acts = [a for v in r.values() for a in v.actions]; order = {"Critical": 0, "Warning": 1, "Info": 2}; acts.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"fleet_health": round(score, 1), "vessels": int(Q("SELECT COUNT(*) n FROM vessels").n[0]), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()}, "critical": sum(a.get("severity") == "Critical" for a in acts)}
    n = f"Fleet health {ev['fleet_health']}/100 with {ev['vessels']} vessels tracked. " + " ".join(f"{k}: {v['summary']}." for k, v in ev["agents"].items())
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": n, "actions": acts, "agents": ev["agents"], "evidence": ev}
def apply(a, c):
    if a["type"] == "reroute":
        sid = a["target"]; v = c.execute("SELECT lat, lon, chain FROM vessels WHERE shipment_id=?", (sid,)).fetchone()
        if v:
            ch = json.loads(v[2]); dest = ch[-1]
            if a.get("opt_idx") is not None:
                import alts; row = c.execute("SELECT s.weight_kg FROM shipments s WHERE shipment_id=?", (sid,)).fetchone(); al = alts.route_alternatives(sys.modules[__name__], (v[0], v[1]), dest, tonnes=float((row or [20000])[0] or 20000) / 1000, interp=interp); i = int(a["opt_idx"]); opt = al[i] if i < len(al) else al[0]
            else: opt = route_options((v[0], v[1]), dest, avoid=("storm",))[0]
            c.execute("UPDATE vessels SET chain=?, progress=0, route_name=? WHERE shipment_id=?", (json.dumps([(v[0], v[1])] + opt["chain"][1:]), ("rerouted: " if a.get("opt_idx") is not None else "rerouted around storm: ") + opt["name"], sid)); return f"vessel re-planned {opt['name']} ({opt['nm']:,} nm)"
    return ""
def decide(action_id, approve, by="user"):
    from live_engine import resolve_action
    with _DB_LOCK, conn() as c:
        t = c.execute("SELECT target FROM agent_actions WHERE action_id=?", (action_id,)).fetchone()
        if t and t[0].startswith("QTE"): c.execute("UPDATE freight_quotes SET status=? WHERE quote_id=?", ("Approved" if approve else "Rejected", t[0]))
    resolve_action(action_id, "approved" if approve else "rejected", f"by {by}")
def submit_request(origin, dest, cargo, weight_t, hs_code, customer_id, language="English", docs_provided=None):
    with _DB_LOCK, conn() as c: return c.execute("INSERT INTO quote_requests(ts,origin,dest,cargo,weight_t,hs_code,customer_id,language,docs_provided) VALUES(?,?,?,?,?,?,?,?,?)", (now(), origin, dest, cargo, weight_t, hs_code, customer_id, language, ";".join(docs_provided) if docs_provided is not None else None)).lastrowid
def feed_status():
    o = time.time() < _OFF[0]; return [("Open-Meteo weather and marine", "Offline: simulated winds in use" if o else "Real · live (cached 15 min)"), ("Vessel positions", "Simulated along real route waypoints (connect AIS via Data Feeder)"), ("Port congestion", "ML predicted, drifts live"), ("Freight-rate model data", "Bootstrap synthetic unless a Kaggle CSV is fed")]
def kpi_extra():
    return [("Vessels tracked", int(Q("SELECT COUNT(*) n FROM vessels").n[0])), ("Quotes awaiting approval", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='Pending Approval'").n[0]))]

def _prewarm():
    try:
        import seanav; seanav.sea()
    except Exception: pass
import threading as _th; _th.Thread(target=_prewarm, daemon=True).start()


In [ ]:
%%writefile /content/freight_backend/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


In [ ]:
%%writefile /content/freight_backend/gpu.py
"""Minimal stub (no language model in this version). Kept so hardening.py can report health."""
def status(): return {"cuda": False, "gpu": None, "models": {}, "stats": {}}


In [ ]:
%%writefile /content/freight_backend/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    if df is None or df.empty: return ""
    try: return df.to_markdown(index=False)
    except: pass
    headers = list(df.columns)
    lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        vals = [str(v) if v is not None else "" for v in row.values]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

INTENT_MAP = {
    "ports": ["port", "ports", "harbor", "terminal", "congestion", "dwell"],
    "shipments": ["shipment", "shipments", "cargo", "carrier", "freight", "teu", "transit"],
    "quotes": ["quote", "spot", "price", "rate", "baf", "margin", "cost"],
    "weather": ["weather", "typhoon", "storm", "cyclone", "wind", "wave"],
    "customs": ["customs", "tariff", "duty", "hs code", "import", "export"],
    "alerts": ["alert", "incident", "delay", "disruption", "hold"]
}

def classify_intent(query):
    q = query.lower()
    for intent, keywords in INTENT_MAP.items():
        if any(kw in q for kw in keywords):
            return intent
    return "general"

def handle_freight_intent(query):
    q_low = query.lower()
    try:
        with get_conn() as conn:
            # 1. Ports / Terminals
            if any(k in q_low for k in ["port", "ports", "harbor", "terminal", "congestion"]):
                tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
                df_ports = pd.read_sql("SELECT port_name as 'Port Name', country as 'Country', region as 'Region', congestion_index as 'Congestion (1-5)', avg_dwell_days as 'Avg Dwell Days' FROM ports ORDER BY congestion_index DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Ports = {tot_ports}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ports)}", "Ports Ledger DB"

            # 2. Shipments / Cargo
            if any(k in q_low for k in ["shipment", "shipments", "carrier", "cargo"]):
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("SELECT shipment_id, origin_port, dest_port, carrier, status, predicted_delay_risk FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Shipments = {tot_shipments}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"

            # 3. Quotes / Margins
            if any(k in q_low for k in ["quote", "quotes", "spot", "margin", "price", "rate"]):
                tot_quotes = conn.execute("SELECT COUNT(*) FROM freight_quotes;").fetchone()[0]
                df_q = pd.read_sql("SELECT quote_id, shipment_id, ROUND(base_cost, 0) AS base_cost_usd, ROUND(final_price, 0) AS final_price_usd, ROUND(margin_pct, 1) AS margin_pct FROM freight_quotes ORDER BY margin_pct DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Active Freight Quotes = {tot_quotes}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_q)}", "Freight Quotes DB"

            # 4. Weather
            if any(k in q_low for k in ["weather", "typhoon", "storm", "wind", "wave", "cyclone", "risk", "risks"]):
                tot_weather = conn.execute("SELECT COUNT(*) FROM weather_risks;").fetchone()[0]
                df_w = pd.read_sql("SELECT port_name AS 'Port Name', current_severity AS 'Severity (1-5)', forecast AS 'Forecast', wind_speed AS 'Wind (Knots)', wave_height AS 'Wave (m)' FROM weather_risks ORDER BY current_severity DESC LIMIT 5;", conn)
                return f"CRITICAL FACT: Total Monitored Weather Risk Hubs = {tot_weather}.\n\n[Sample Subset Preview - First 5 Records Only]:\n{df_to_markdown_safe(df_w)}", "Weather Risks DB"
    except Exception:
        pass
    return None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_freight as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    freight = handle_freight_intent(query)
    if freight:
        return freight

    try:
        from rag_engine import answer_with_citation
        ctx, src = answer_with_citation(query)
        return ctx, src
    except Exception:
        return f"Retrieved enterprise freight intelligence for query: '{query}'.", "General Knowledge Index"

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)


In [ ]:
%%writefile /content/freight_backend/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    return {}      # no approval queue in this version: agents act on their own
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}
    if levels: return LADDER[:1] + (["Admin"] if int(levels) >= 2 else [])
    if typ in cfg and cfg[typ]: return list(cfg[typ])
    return ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,status,note,approval) VALUES(?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "manual_request", levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT


In [ ]:
%%writefile /content/freight_backend/mailer.py
"""One place that sends e-mail (OTP codes, reports, alerts). Gmail SMTP with an app password: set EMAIL_ID + EMAIL_PASSWORD (Colab secrets or env)."""
import os, ssl, smtplib
from email.message import EmailMessage
def creds(): return os.environ.get("EMAIL_ID") or "", (os.environ.get("EMAIL_PASSWORD") or os.environ.get("EMAIL_APP_PASSWORD") or "").replace(" ", "")
def configured(): a, b = creds(); return bool(a and b)
def send(to, subject, body, attachments=(), html=None):
    """attachments: [(filename, bytes, mimetype)]. Returns (ok, message)."""
    em, pw = creds()
    if not (em and pw): return False, "E-mail is not configured (set EMAIL_ID and EMAIL_PASSWORD)."
    try:
        m = EmailMessage(); m["Subject"] = subject; m["From"] = em; m["To"] = to; m.set_content(body)
        if html: m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments: m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s: s.login(em, pw); s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e: return False, f"Could not send: {e}"


In [ ]:
%%writefile /content/freight_backend/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re
from blocks import Q

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def answer(kind, q, scope=None, mask=False):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


In [ ]:
%%writefile /content/freight_backend/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))


In [ ]:
%%writefile /content/freight_backend/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


In [ ]:
%%writefile /content/freight_backend/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas"),
    ("Ningbo-Zhoushan Port", "China", 3.9, 5, 29.87, 121.55, "Asia"),
    ("Shenzhen Port", "China", 3.7, 4, 22.54, 114.05, "Asia"),
    ("Guangzhou Port", "China", 3.3, 4, 23.10, 113.30, "Asia"),
    ("Qingdao Port", "China", 3.4, 4, 36.07, 120.38, "Asia"),
    ("Tianjin Port", "China", 3.6, 4, 39.00, 117.72, "Asia"),
    ("Hong Kong Port", "Hong Kong", 2.1, 3, 22.29, 114.16, "Asia"),
    ("Kaohsiung Port", "Taiwan", 2.4, 3, 22.62, 120.28, "Asia"),
    ("Laem Chabang Port", "Thailand", 2.6, 3, 13.08, 100.88, "Asia"),
    ("Port Klang", "Malaysia", 2.0, 3, 3.00, 101.39, "Asia"),
    ("Tanjung Pelepas Port", "Malaysia", 1.7, 2, 1.36, 103.55, "Asia"),
    ("Jakarta Tanjung Priok", "Indonesia", 3.3, 4, -6.10, 106.88, "Asia"),
    ("Manila Port", "Philippines", 3.5, 4, 14.58, 120.95, "Asia"),
    ("Ho Chi Minh Cat Lai Port", "Vietnam", 3.0, 4, 10.76, 106.77, "Asia"),
    ("Karachi Port", "Pakistan", 3.8, 5, 24.85, 66.98, "Asia"),
    ("Bandar Abbas Port", "Iran", 3.6, 4, 27.19, 56.28, "Middle East"),
    ("Jeddah Islamic Port", "Saudi Arabia", 2.2, 3, 21.49, 39.17, "Middle East"),
    ("Salalah Port", "Oman", 1.8, 2, 17.02, 54.09, "Middle East"),
    ("Piraeus Port", "Greece", 2.3, 3, 37.94, 23.65, "Europe"),
    ("Valencia Port", "Spain", 2.1, 3, 39.44, -0.32, "Europe"),
    ("Algeciras Port", "Spain", 2.0, 2, 36.13, -5.45, "Europe"),
    ("Felixstowe Port", "UK", 2.8, 3, 51.96, 1.35, "Europe"),
    ("Le Havre Port", "France", 2.4, 3, 49.49, 0.11, "Europe"),
    ("Gdansk Port", "Poland", 2.2, 3, 54.35, 18.65, "Europe"),
    ("Genoa Port", "Italy", 2.6, 3, 44.41, 8.93, "Europe"),
    ("Durban Port", "South Africa", 3.4, 4, -29.87, 31.02, "Africa"),
    ("Tangier Med Port", "Morocco", 1.9, 2, 35.88, -5.50, "Africa"),
    ("Lagos Apapa Port", "Nigeria", 4.0, 5, 6.45, 3.37, "Africa"),
    ("New York/New Jersey Port", "USA", 2.9, 3, 40.68, -74.02, "Americas"),
    ("Savannah Port", "USA", 2.7, 3, 32.08, -81.09, "Americas"),
    ("Long Beach Port", "USA", 3.4, 4, 33.75, -118.19, "Americas"),
]


def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # User authentication is handled by the Milestone 1 Streamlit database.

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"

            # Generate the exact 9 fields expected by the database
            weight = round(random.uniform(500.0, 45000.0), 1)
            eta_val = (datetime.date.today() + datetime.timedelta(days=random.randint(2, 30))).isoformat()
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            hs = f"HS-{random.randint(8400, 8900)}"

            # Insert matching the 9 columns exactly
            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, eta, weight_kg, hs_code, predicted_delay_risk) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), eta_val, weight, hs, d_risk)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")

        # ADD THE MISSING LIST HERE:
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()


In [ ]:
%%writefile /content/freight_backend/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE

def can(c, cap): return True      # one operator, no roles in this version
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    return df[df[col].astype(str).str.startswith(c["scope"])] if c.get("scope") and len(df) else df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    """Read-only list of what the agent has raised (type, target, severity, detail). There is no approval workflow in this version."""
    df = tk_df(c, agent, types, target, limit=limit)
    if not len(df): return B.callout("Nothing raised right now.", "info", title)
    return B.table(df, ["id", "ts", B.pill_col("severity"), "type", "target", "detail"], title=title, h=300, key=key or f"tk_{agent or target or 'all'}", open={"col": "target"} if False else None)
def hero(c, agent, title, blurb): return B.hero(agent, title, blurb, latest(agent), can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}





# ------------------------------------------------------------------ Knowledge graph
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e or opts[0]
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


In [ ]:
%%writefile /content/freight_backend/views_freight.py
"""FreightQuote AI pages (server-driven blocks)."""
import os, re, json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import domain_freight as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FreightQuote AI"
MENU = [('copilot', 'AI Copilot', 'bot'), ('route', 'Agent 1 · Route AI', 'route'), ('quotes', 'Agent 2 · Spot Quotes', 'dollar'), ('carriers', 'Agent 3 · Carriers', 'truck'), ('weather', 'Agent 4 · Weather Risk', 'cloud'), ('margin', 'Agent 5 · Margin', 'trend'), ('customs', 'Agent 6 · Customs', 'file')]
SEVC = {2: RED, 1: AMBER, 0: BLUE, -1: TEAL}
FLEET_LEGEND = [("Vessel · no ticket", TEAL), ("Vessel · info ticket", BLUE), ("Vessel · warning ticket", AMBER), ("Vessel · critical ticket", RED), ("Port · low congestion (big dot)", "#8b8d98"), ("Port · congestion 3+", "#d98e04"), ("Port · congestion 4+", "#b4262b"), ("Storm cell (circle)", "#e5484d")]
def dport(ports, key, alt=0): return next((p for p in ports if key.lower() in p.lower()), ports[min(alt, len(ports) - 1)])

# ------------------------------------------------------------------ map layers
def fleet_layers(vessels=True, ports=True, storms=True, v=None):
    layers = []
    if ports:
        p = Q("SELECT port_name, lat, lon, congestion_index ci FROM ports"); layers.append({"type": "points", "id": "ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": "#b4262b" if r.ci >= 4 else "#d98e04" if r.ci >= 3 else "#8b8d98", "r": 3 + r.ci, "tip": f"<b>{r.port_name}</b><br>congestion {r.ci:.1f}", "open": r.port_name, "z": 1} for r in p.itertuples()]})
    if vessels:
        v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, v.progress, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)") if v is None else v
        tk = Q("SELECT target, MAX(CASE severity WHEN 'Critical' THEN 2 WHEN 'Warning' THEN 1 ELSE 0 END) sv FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").sv
        layers.append({"type": "points", "id": "vessels", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": SEVC[int(tk.get(r.shipment_id, -1))], "r": 5, "tip": f"<b>{r.shipment_id}</b><br>{r.o} → {r.d}<br>{r.route_name} · {r.progress:.0%}", "open": r.shipment_id, "z": 2} for r in v.itertuples()]})
    if storms: layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); oa = tk_df(c, limit=500); ex = dom.kpi_extra(); fl = Q("SELECT COUNT(*) n FROM vessels").n[0]
    mp = B.map_(fleet_layers(), h=510, view={"center": [20, 60], "zoom": 2}, key="ops_map", legend=FLEET_LEGEND)
    q = Q("SELECT quote_id, ROUND(final_price) price, margin_pct margin, status FROM freight_quotes ORDER BY rowid DESC LIMIT 7"); p = Q("SELECT port_name, ROUND(congestion_index,1) ci FROM ports ORDER BY congestion_index DESC LIMIT 7")
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; by = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int)
    bl = [B.kpis(("Fleet health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Vessels tracked", int(fl)), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), ("Active storms", len(dom.storms()), None, "Critical" if dom.storms() else None))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Live fleet · click a vessel or port", mp), B.col(B.card("Latest quotes", B.table(q, ["quote_id", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status")], h=230, open={"col": "quote_id"}, key="lq")), B.card("Most congested ports", B.table(p, ["port_name", B.num_col("ci", "index", "bar", 5)], h=230, open={"col": "port_name"}, key="cp"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=220, key="trend")), B.card("Open tickets by agent", B.chart(B.bar(by.index, by.values, horizontal=True, colors=[BLUE]), h=220, key="byag"))))
    if can(c, "admin"): bl.append(B.button("storm", "Inject a demo storm (tests the re-route loop)", "/api/freight/storm", "ghost", payload={}))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k == "shipment":
        v = Q("SELECT v.lat, v.lon, v.route_name, v.progress, v.chain, s.origin_port o, s.dest_port d, s.carrier, s.status, s.eta, s.weight_kg, s.hs_code, s.predicted_delay_risk risk FROM vessels v RIGHT JOIN shipments s USING(shipment_id) WHERE s.shipment_id=?", (e,))
        if not len(v): return {"title": e, "blocks": [B.callout("Unknown shipment.", "warn")]}
        r = v.iloc[0]; bl = []
        if pd.notna(r.lat):
            w = dom.weather_at(float(r.lat), float(r.lon), allow_live=False); ch = json.loads(r.chain) if r.chain else []
            bl.append(B.kpis(("Progress", f"{r.progress * 100:.0f}%"), ("Wind m/s", round(w["wind"], 1)), ("Wave m", round(w["wave"], 1)), ("Weather", w["level"], w["src"], TONE.get(w["level"]))))
            bl.append(B.map_([{"type": "lines", "lines": [{"pts": ch, "color": TEAL, "w": 3, "dash": "6 6"}]}, {"type": "points", "id": "me", "pts": [{"id": e, "lat": r.lat, "lon": r.lon, "color": "#1f2430", "r": 8, "tip": e}]}, {"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": "storm"} for s in dom.storms()]}], h=230, view={"center": [float(r.lat), float(r.lon)], "zoom": 3}, key="d_map", legend=[("This port", "#1f2430"), ("Planned voyage route", TEAL), ("Storm cell (red circle)", RED)]))
        else: bl.append(B.callout(f"Not at sea: status {r.status}.", "info"))
        bl.append(B.table([{"field": "Route", "value": f"{r.o} → {r.d}"}, {"field": "Carrier", "value": r.carrier}, {"field": "Status", "value": r.status}, {"field": "ETA", "value": r.eta}, {"field": "Cargo weight", "value": f"{(r.weight_kg or 0) / 1000:,.1f} t"}, {"field": "HS code", "value": r.hs_code}, {"field": "Predicted delay risk", "value": r.risk}], ["field", "value"], h=250, key="d_f"))
        bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
        try:
            import alts; po, pd_ = dom.port_xy(r.o), dom.port_xy(r.d); start = (float(r.lat), float(r.lon)) if pd.notna(r.lat) else (po[0], po[1]) if po else None
            if start and pd_:
                ao = alts.route_alternatives(dom, start, (pd_[0], pd_[1]), tonnes=float(r.weight_kg or 20000) / 1000, interp=dom.interp, dwell=(pd_[3] if pd_ else 0) * .5); bl += alts.route_blocks(ao, e, "Route alternatives" + (" from the vessel's position" if pd.notna(r.lat) else ""))
                if can(c, "approve") and pd.notna(r.lat) and ao: bl.append(B.form("rr_" + e, [{"name": "opt", "label": "Switch to", "type": "select", "options": [f"{i + 1}. {x['label']} · {x['name']}" for i, x in enumerate(ao)], "value": "1. " + ao[0]["label"] + " · " + ao[0]["name"]}, {"name": "shipment", "type": "hidden", "value": e}], "Re-route this voyage", "/api/freight/reroute", title="Apply a route", sub="Needs approve rights. The vessel is re-planned from its current position and the change is audited.", tone="ghost"))
        except Exception as ex: bl.append(B.callout(f"Route alternatives unavailable: {ex}", "info"))
        return {"title": e, "sub": f"{r.o} → {r.d}", "blocks": bl}
    if k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if not len(q): return {"title": e, "blocks": [B.callout("Unknown quote.", "warn")]}
        r = q.iloc[0]; cost = r.base_cost + r.fuel_surcharge + r.customs_fee + r.insurance
        return {"title": e, "sub": r.status, "blocks": [B.kpis(("Price", f"${r.final_price:,.0f}"), ("Margin", f"{r.margin_pct}%"), ("Status", r.status)), B.chart(B.waterfall(["Base (ML)", "Fuel", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel_surcharge, r.customs_fee, r.insurance, r.final_price - cost]), "How the price is built", h=260, key="d_wf"), ticket_block(c, target=e, title="Tickets", key="d_tk")]}
    p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
    if not len(p): return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    r = p.iloc[0]; w = Q("SELECT current_severity sev, wind_speed, wave_height FROM weather_risks WHERE port_name=?", (e,)); sh = Q("SELECT shipment_id, carrier, status FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 12", (e, e))
    bl = [B.kpis(("Congestion index", round(float(r.congestion_index), 2), None, "Critical" if r.congestion_index >= 4 else None), ("Avg dwell days", r.avg_dwell_days), ("Country", r.country)), B.table(sh, ["shipment_id", "carrier", B.pill_col("status")], title="Shipments via this port", h=250, open={"col": "shipment_id"}, key="d_sh")]
    try:
        import alts; bl += alts.port_alternatives(e)
    except Exception: pass
    if len(w): bl.insert(1, B.kpis(("Weather", w.sev[0], f"wind {w.wind_speed[0]:.0f} km/h · wave {w.wave_height[0]:.1f} m", TONE.get(w.sev[0]))))
    return {"title": e, "sub": f"{r.country} · port", "blocks": bl}

# ------------------------------------------------------------------ pages
def route(c, tab=None):
    v = Q("SELECT COUNT(*) n FROM vessels").n[0]; p = Q("SELECT port_name, country, region, congestion_index ci, avg_dwell_days dw FROM ports"); sh = Q("SELECT status, COUNT(*) n FROM shipments GROUP BY status").set_index("status").n
    ports = sorted(p.port_name); pr = c.get("params") or {}; o = pr.get("o") or dport(ports, "JNPT"); d = pr.get("d") or dport(ports, "Rotterdam", 1); cg = pr.get("cargo") or "Electronics"
    bl = [hero(c, "route_watch", "Route & congestion agent", "Compares Suez and Cape routings with a RandomForest congestion classifier and a live weather check along the lane. Watches every port."),
          B.kpis(("Vessels tracked", int(v)), ("Ports monitored", len(p)), ("Congested (≥ 4)", int((p.ci >= 4).sum()), None, "Warning"), ("Active storms", len(dom.storms())))]
    bl.append(B.card("Live fleet", B.map_(fleet_layers(), h=400, view={"center": [20, 60], "zoom": 2}, key="rt_map", legend=FLEET_LEGEND)))
    top = p.sort_values("ci", ascending=False).head(12)
    bl.append(B.row(B.card("Most congested ports (click)", B.chart(B.bar(top.port_name, top.ci.round(2), horizontal=True, itemcolors=[RED if x >= 4 else AMBER for x in top.ci], max_=5), h=330, open={"bar": "y"}, key="rt_c")), B.card("Shipments by status", B.chart(B.bar(sh.index, sh.values.astype(int), colors=[BLUE]), h=330, key="rt_s"))))
    bl.append(B.form("rt_plan", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": o}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": d}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": cg}], "Plan route", "page:route", title="Route planner", sub="Same route, weather and congestion logic as the quote pipeline.", tone="ghost"))
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if po and pd_ and o != d:
        opts = dom.route_options(po[:2], pd_[:2]); cong = dom.predict_congestion((po[2] + pd_[2]) / 2, (po[3] + pd_[3]) / 2, cg); rows, lines = [], []
        for i, op in enumerate(opts):
            ws = [dom.weather_at(*dom.interp(op["chain"], f), allow_live=False) for f in (0, .25, .5, .75, 1)]; worst = max(ws, key=lambda w: w["wind"]); days = op["nm"] / (14 * 24) + po[3] * .5 + pd_[3] * .5
            rows.append({"route": op["name"], "distance_nm": op["nm"], "transit_days": round(days, 1), "weather_risk": worst["level"], "max_wind_ms": worst["wind"]}); lines.append({"pts": op["chain"], "color": TEAL if i == 0 else AMBER, "w": 4, "dash": None if i == 0 else "8 8", "tip": op["name"]})
        bl.append(B.row(B.card("Route options", B.table(rows, ["route", B.num_col("distance_nm", "nm", "int"), B.num_col("transit_days", "days", "num"), B.pill_col("weather_risk"), B.num_col("max_wind_ms", "max wind m/s", "num")], h=150, key="rt_o"), B.callout(f"Predicted congestion level {cong}/3 (RandomForest: port congestion index, dwell time, month, weekday, cargo).", "info")),
                        B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "ends", "pts": [{"id": o, "lat": po[0], "lon": po[1], "color": TEAL, "r": 7, "tip": o}, {"id": d, "lat": pd_[0], "lon": pd_[1], "color": BLUE, "r": 7, "tip": d}]}], h=300, key="rt_pm", view={"bounds": [[min(p[0] for ln in lines[:1] for p in ln["pts"]) - 4, min(p[1] for ln in lines[:1] for p in ln["pts"]) - 4], [max(p[0] for ln in lines[:1] for p in ln["pts"]) + 4, max(p[1] for ln in lines[:1] for p in ln["pts"]) + 4]]}, legend=[("Recommended (sea route)", TEAL), ("Alternative", AMBER)])))
    x = cross(c, p[p.ci >= 4].port_name.tolist(), "route_watch"); bl += [x] if x else []
    bl.append(ticket_block(c, "route_watch", title="Open tickets from this agent", key="tk_rt")); return B.page("Route AI", "Agent 1 · routing, congestion, lane planning", bl, live=10)

def _docs_all(): return sorted({d for s in Q("SELECT docs FROM customs_rules").docs for d in str(s).split(";") if d})
def quote_form(c):
    ports = Q("SELECT port_name FROM ports ORDER BY port_name").port_name.tolist(); cust = Q("SELECT customer_id FROM customers").customer_id.tolist(); alld = _docs_all(); ch, base, *_ = dom._required_docs("HS-8501")
    return B.form("quote", [{"name": "o", "label": "Origin", "type": "select", "options": ports, "value": dport(ports, "JNPT")}, {"name": "d", "label": "Destination", "type": "select", "options": ports, "value": dport(ports, "Rotterdam", 1)}, {"name": "cargo", "label": "Cargo", "type": "select", "options": list(dom.CARGO), "value": "Electronics"},
                                  {"name": "weight", "label": "Weight (tons)", "type": "number", "value": 120, "min": 1, "max": 5000}, {"name": "hs", "label": "HS code", "type": "text", "value": "HS-8501"}, {"name": "customer", "label": "Customer", "type": "select", "options": cust, "value": cust[0]},
                                  {"name": "docs", "label": "Documents you are submitting (tick all that apply; the customs agent checks them against the rules for the HS code)", "type": "checks", "options": alld, "value": base}], "Generate quote", "/api/freight/quote", title="New quote", sub="Runs the 7-agent pipeline: route → weather → (re-plan loop) → pricing (RandomForest) → customs → margin → documents → alerts.")
def quote_result(rid):
    r = Q("SELECT quote_id, result FROM quote_requests WHERE req_id=?", (rid,))
    if not len(r) or not r.result[0]: return [B.callout("The pipeline did not return a result.", "warn")]
    res = json.loads(r.result[0]); bl = [B.callout(res.get("narrative", ""), "success", "Pipeline result"), B.kpis(("Price (USD)", f"{res.get('price', 0):,.0f}"), ("Decision", res.get("decision")), ("Margin", f"{res.get('margin_pct')}%"), ("Transit", f"{res.get('days')} days"))]
    if res.get("missing"): bl.append(B.callout("Missing documents: " + ", ".join(res["missing"]) + ". Attach them on the Customs page and the quote re-runs.", "warn"))
    tr = Q("SELECT agent, risk, score, latency_ms ms, summary FROM agent_runs WHERE cycle_id LIKE ? ORDER BY run_id", (f"Q{rid}-%",)); bl.append(B.table(tr, ["agent", B.pill_col("risk"), B.num_col("score", "score", "num"), B.num_col("ms", "ms", "int"), "summary"], title="Agent trace for this quote", h=260, key="q_tr"))
    ddir = os.path.join(dom.DATA_DIR, "docs"); files = [p for p in (sorted(os.listdir(ddir)) if os.path.isdir(ddir) else []) if p.startswith(r.quote_id[0])]
    for p in files: bl.append(B.button("dl_" + p, "Download " + p, f"/api/doc/{p}", "ghost", payload={"download": True}))
    return bl
def quotes(c, tab=None):
    bl = [hero(c, "quote_watch", "Spot quotes agent", "Every quote goes through one pipeline: ML price (RandomForest) + 12% fuel + port fees + 15% operating cost, target margin ~20%. Quotes above $50,000 need admin approval.")]
    if can(c, "quote"): bl.append(quote_form(c))
    if can(c, "view"):
        q = Q("SELECT quote_id, customer_id, carrier, final_price price, margin_pct margin, status, created_at FROM freight_quotes ORDER BY created_at DESC, quote_id DESC LIMIT 400"); st = q.status.value_counts()
        bl.append(B.kpis(("Quotes", len(q)), ("Approved", int((q.status == "Approved").sum()), None, "Low"), ("Pending approval", int((q.status == "Pending Approval").sum()), None, "Warning"), ("On hold", int((q.status == "On Hold").sum()), None, "Critical")))
        bl.append(B.row(B.card("Quote decisions", B.chart(B.pie(st.to_dict(), colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=280, key="qt_p")), B.card("Margin % across quotes", B.chart(B.hist(q.margin, 24, BLUE), h=280, key="qt_h"))))
        bl.append(B.table(q.head(30), ["quote_id", "customer_id", "carrier", B.num_col("price", "price", "money"), B.num_col("margin", "margin %", "num"), B.pill_col("status"), "created_at"], title="Latest quotes · click a row", h=340, open={"col": "quote_id"}, key="qt_t", search=True))
        bl.append(ticket_block(c, "quote_watch", title="Open tickets from this agent", key="tk_qt"))
    return B.page("Spot Quotes", "Agent 2 · the 7-agent quote pipeline", bl, live=10)

def carriers(c, tab=None):
    k = Q("SELECT name, rating, on_time_pct, avg_cost_index, risk_level FROM carriers"); k["score"] = (k.rating / 5 * 40 + k.on_time_pct * .4 - k.avg_cost_index * 12 - (k.risk_level == "High") * 8).round(1)
    s = Q("SELECT carrier name, COUNT(*) shipments, SUM(status LIKE '%Delay%' OR status='Customs Hold') problems, AVG(predicted_delay_risk) delay_risk FROM shipments GROUP BY carrier"); k = k.merge(s, on="name", how="left").fillna({"shipments": 0, "problems": 0})
    top = k.sort_values("score", ascending=False).head(12); rc = {"Low": TEAL, "Medium": AMBER, "High": RED}
    sc = {t: [{"x": a.on_time_pct, "y": a.avg_cost_index, "label": a.name, "size": 6 + a.shipments ** .5} for a in k[k.risk_level == t].itertuples()] for t in ["Low", "Medium", "High"] if (k.risk_level == t).any()}
    bl = [hero(c, "route_watch", "Carrier intelligence", "Score = rating × 40% + on-time % × 0.4 − cost-index penalty − 8 points for High risk. Shipments and delays come from the live shipments table."),
          B.kpis(("Carriers", len(k)), ("High-risk carriers", int((k.risk_level == "High").sum()), None, "Warning"), ("Avg on-time", f"{k.on_time_pct.mean():.1f}%"), ("Best score", k.sort_values("score").name.iloc[-1])),
          B.row(B.card("Top carriers by agent score", B.chart(B.bar(top.name, top.score, horizontal=True, colors=[TEAL], sort=True), h=360, key="ca_t")), B.card("On-time vs cost index (size = shipments)", B.chart(B.scatter(sc, "on-time %", "cost index", colors=rc, sized=True), h=360, key="ca_s"))),
          B.table(k.sort_values("score", ascending=False), ["name", B.num_col("score", "score", "bar", 100), "rating", B.num_col("on_time_pct", "on-time %", "num"), B.num_col("avg_cost_index", "cost idx", "num"), B.pill_col("risk_level", "risk"), B.num_col("shipments", "shipments", "int"), B.num_col("problems", "problems", "int")], title="Carrier league table", h=340, key="ca_tb", search=True)]
    return B.page("Carriers", "Agent 3 · ranking and risk", bl, live=15)

METRICS = {"wind": ("Wind speed", [("Calm / light  < 8 m/s", TEAL), ("Strong  8–15 m/s", AMBER), ("Storm-force  > 15 m/s", RED)]),
           "waves": ("Wave height", [("Slight  < 2.5 m", TEAL), ("Rough  2.5–4 m", AMBER), ("Very rough  > 4 m", RED)]),
           "rain": ("Rain now", [("Dry", "#8b8d98"), ("Light rain  < 2.5 mm/h", BLUE), ("Heavy rain  ≥ 2.5 mm/h", VIOLET)]),
           "temp": ("Temperature", [("Cold  < 10 °C", BLUE), ("Mild  10–30 °C", TEAL), ("Hot  30–35 °C", AMBER), ("Very hot  > 35 °C", RED)])}
def _wxcolor(metric, r):
    if metric == "wind": return RED if r.wind_ms > 15 else AMBER if r.wind_ms >= 8 else TEAL
    if metric == "waves": w = r.wave_m if r.wave_m == r.wave_m and r.wave_m is not None else 0; return RED if w > 4 else AMBER if w >= 2.5 else TEAL
    if metric == "rain": p = r.precip_mm or 0; return VIOLET if p >= 2.5 else BLUE if p > 0 else "#8b8d98"
    t = r.temp_c if r.temp_c is not None else 25; return RED if t > 35 else AMBER if t >= 30 else TEAL if t >= 10 else BLUE
def wx_layers(metric, hi):
    import weather as WX
    t = WX.table(); layers = []
    if len(t):
        pts = []
        for r in t.itertuples():
            w = r.wave_m if r.wave_m is not None and r.wave_m == r.wave_m else None
            pts.append({"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": _wxcolor(metric, r), "r": 4 + min(9, r.wind_ms / 2.2), "z": 1, "open": r.port_name,
                        "tip": f"<b>{r.port_name}</b> · {WX.WMO.get(r.wcode, '')}<br>wind {r.wind_ms:.1f} m/s ({r.wind_ms * 3.6:.0f} km/h), gusts {r.gust_ms or 0:.0f} m/s<br>waves {'-' if w is None else f'{w:.1f} m'} · rain {r.precip_mm or 0:.1f} mm/h · {r.temp_c:.0f} °C<br><i>real reading {r.ts[11:16]} UTC</i>"})
        layers.append({"type": "points", "id": "wx_ports", "pts": pts})
    else:
        w = Q("SELECT w.port_name, w.current_severity sev, w.wind_speed, w.wave_height, w.temperature, p.lat, p.lon FROM weather_risks w JOIN ports p USING(port_name)")
        layers.append({"type": "points", "id": "wx_ports", "pts": [{"id": r.port_name, "lat": r.lat, "lon": r.lon, "color": RED if r.sev == "High" else AMBER if r.sev == "Medium" else TEAL, "r": 4 + min(9, r.wind_speed / 8), "z": 1, "open": r.port_name, "tip": f"<b>{r.port_name}</b><br>wind {r.wind_speed:.0f} km/h · waves {r.wave_height:.1f} m<br><i>simulated, not a real reading</i>"} for r in w.itertuples()]})
    v = Q("SELECT v.shipment_id, v.lat, v.lon, v.route_name, s.origin_port o, s.dest_port d FROM vessels v JOIN shipments s USING(shipment_id)"); hs = set(hi)
    layers.append({"type": "points", "id": "wx_v", "pts": [{"id": r.shipment_id, "lat": r.lat, "lon": r.lon, "color": "#1f2430" if r.shipment_id in hs else "#6b7080", "r": 4.5 if r.shipment_id in hs else 3, "z": 2, "open": r.shipment_id, "tip": f"<b>{r.shipment_id}</b> {r.o} → {r.d}" + ("<br><b>in storm-force wind</b>" if r.shipment_id in hs else "")} for r in v.itertuples()]})
    layers.append({"type": "circles", "items": [{"lat": s["lat"], "lon": s["lon"], "r_m": float(s.get("r", 700)) * 1852, "color": RED, "tip": f"Storm cell · wind {s.get('wind', '?')} m/s"} for s in dom.storms()]})
    return layers
def weather(c, tab=None):
    import weather as WX
    p = c.get("params") or {}; metric = p.get("metric") if p.get("metric") in METRICS else "wind"; st = WX.status()
    r = (latest("weather_watch") or {}).get("out", {}); hi = r.get("vessels_high", []); w = Q("SELECT port_name, current_severity sev, wind_speed, wave_height, temperature, forecast FROM weather_risks")
    src = (f"Real · Open-Meteo, {st['ok']}/{st['total']} ports, updated {st['ts'][11:16]} UTC ({st['age_min']:.0f} min ago)" if st["live"] else "Simulated · Open-Meteo not reachable" + (f" ({st['err']})" if st["err"] else "") + ". Press Refresh to retry.")
    lg = list(METRICS[metric][1]) + [("Big dot = port (size = wind)", "#8b8d98"), ("Black dot = vessel in storm-force wind", "#1f2430"), ("Grey dot = other vessel", "#6b7080"), ("Red circle = storm cell", "#e5484d")]
    pick = {"t": "form", "id": "wx_pick", "fields": [{"name": "metric", "label": "Colour ports by", "type": "select", "options": list(METRICS), "value": metric}], "submit": "Show", "url": "page:weather", "title": None, "tone": "ghost", "inline": True}
    hot = w.sort_values("wind_speed", ascending=False)
    bl = [B.callout(src, "success" if st["live"] else "warning", "Weather data source"), hero(c, "weather_watch", "Weather risk agent", "Real wind, wave, rain and temperature from Open-Meteo for every port (refreshed every 20 min) plus injected storm cells. Vessels in High risk (wind > 15 m/s) get a re-route ticket; the system re-plans the voyage once it is approved."),
          B.kpis(("Vessels in High risk", len(hi), None, "Critical" if hi else None), ("Active storms", len(dom.storms())), ("Ports with weather data", len(w)), ("Ports in High risk", int((w.sev == "High").sum()), None, "Warning"), ("Data", "Real" if st["live"] else "Simulated", None, "Low" if st["live"] else "Warning")),
          pick, B.card(f"Port weather · coloured by {METRICS[metric][0].lower()} · click a port or vessel", B.map_(wx_layers(metric, hi), h=470, view={"center": [20, 60], "zoom": 2}, key="wx_map", legend=lg)),
          B.row(B.card("Strongest port winds (km/h)", B.chart(B.bar(hot.head(12).port_name, hot.head(12).wind_speed.round(0), horizontal=True, colors=[AMBER], sort=True), h=330, open={"bar": "y"}, key="wx_w")), B.card("Port weather risk levels", B.chart(B.pie({str(k): int(v) for k, v in w.sev.value_counts().items()}, colors={"High": RED, "Medium": AMBER, "Low": TEAL}), h=330, key="wx_d")))]
    if hi: bl.append(B.table(Q("SELECT shipment_id, lat, lon, route_name FROM vessels WHERE shipment_id IN (%s)" % ",".join("?" * len(hi[:50])), hi[:50]), ["shipment_id", B.num_col("lat", "lat", "num"), B.num_col("lon", "lon", "num"), "route_name"], title="Vessels in High-risk weather now · click to inspect", h=240, open={"col": "shipment_id"}, key="wx_hv"))
    bl.append(B.table(w.sort_values("wind_speed", ascending=False).head(60).rename(columns={"wind_speed": "wind km/h", "wave_height": "waves m", "temperature": "temp °C"}), ["port_name", B.pill_col("sev"), "wind km/h", "waves m", "temp °C", "forecast"], title="Port weather (real where the forecast says Open-Meteo live)", h=300, open={"col": "port_name"}, key="wx_t"))
    if can(c, "run"): bl.append(B.button("wx_ref", "Refresh real weather now", "/api/weather/refresh", "ghost", payload={}))
    bl.append(B.table(pd.DataFrame(dom.feed_status(), columns=["feed", "status"]), title="Data sources", h=190, key="wx_f"))
    if can(c, "admin"):
        PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; bl.append(B.form("storm", [{"name": "area", "label": "Area", "type": "select", "options": list(PRESET_STORMS), "value": "Red Sea"}, {"name": "wind", "label": "Wind m/s", "type": "range", "min": 15, "max": 35, "value": 24}], "Inject storm", "/api/freight/storm", title="Inject a storm", sub="Adds a storm cell for testing the re-route loop; it does not change the real readings."))
    bl.append(ticket_block(c, "weather_watch", title="Open tickets from this agent", key="tk_wx")); return B.page("Weather Risk", "Agent 4 · real wind, waves, rain, storms", bl, live=10)

def margin(c, tab=None):
    q = Q("SELECT quote_id, customer_id, base_cost, fuel_surcharge fuel, customs_fee, insurance, final_price price, margin_pct margin, status FROM freight_quotes WHERE base_cost>0 ORDER BY created_at DESC, quote_id DESC LIMIT 600")
    bl = [hero(c, "quote_watch", "Margin & approvals agent", "Margin = (price − cost) ÷ price with tiered targets (Platinum 18%, Gold 20%, Silver/Bronze 22–24%). Quotes > $50,000 or < 10% margin wait for an approver."),
          B.kpis(("Average margin", f"{q.margin.mean():.1f}%"), ("Below 10% margin", int((q.margin < 10).sum()), None, "Warning"), ("Over $50,000", int((q.price > 50000).sum())), ("Revenue in quotes", f"${q.price.sum():,.0f}"))]
    sc = {s: [{"x": a.base_cost, "y": a.price, "label": a.quote_id} for a in q[q.status == s].head(250).itertuples()] for s in q.status.unique()}; cm = q.groupby("customer_id").margin.mean().round(1).sort_values().head(10)
    bl.append(B.row(B.card("Price vs base cost (click a dot)", B.chart(B.scatter(sc, "base cost", "price", colors={"Approved": TEAL, "Pending Approval": AMBER, "On Hold": RED, "Rejected": "#8b8d98"}), h=330, open={"scatter": True}, key="mg_s")), B.card("Lowest-margin customers", B.chart(B.bar(cm.index, cm.values, horizontal=True, colors=[AMBER]), h=330, key="mg_c"))))
    if len(q):
        sel = (c.get("params") or {}).get("q") or q.quote_id.iloc[0]; r = q[q.quote_id == sel]; r = r.iloc[0] if len(r) else q.iloc[0]; cost = r.base_cost + r.fuel + r.customs_fee + r.insurance
        bl.append({"t": "form", "id": "mg_pick", "fields": [{"name": "q", "label": "Cost build-up for quote", "type": "select", "options": q.quote_id.tolist()[:200], "value": r.quote_id}], "submit": "Show", "url": "page:margin", "title": None, "tone": "ghost", "inline": True})
        bl.append(B.card(f"{r.quote_id}: how the price is built", B.chart(B.waterfall(["Base (ML)", "Fuel 12%", "Customs", "Insurance", "Margin"], [r.base_cost, r.fuel, r.customs_fee, r.insurance, r.price - cost]), h=300, key="mg_w")))
    bl.append(ticket_block(c, "quote_watch", ["quote_approval", "margin_review"], title="Margin actions", key="tk_mg")); return B.page("Margin", "Agent 5 · margin and pricing tiers", bl, live=10)

def customs(c, tab="customs"):
    tabs = [{"id": "customs", "label": "Customs & tariff"}, {"id": "docs", "label": "Documents (OCR)"}]
    if tab == "docs":
        ddir = os.path.join(dom.DATA_DIR, "docs"); files = sorted(os.listdir(ddir), reverse=True) if os.path.isdir(ddir) else []
        bl = [B.callout("PDFs are generated by the Document agent for every quote (Bill of Lading, Packing List). Upload a document to read its fields and attach it to the matching quote.", "info"), B.kpis(("Generated documents", len(files)))]
        bl += [B.button("dl_" + f, "Download " + f, f"/api/doc/{f}", "ghost", payload={"download": True}) for f in files[:10]]
        bl.append(B.form("readdoc", [{"name": "file", "label": "PDF or TXT", "type": "file"}], "Read document", "/api/freight/readdoc", title="Read an uploaded document (OCR / text extraction)", tone="primary")); return B.page("Customs", "Agent 6 · documents", bl, tabs, live=0)
    h = Q("SELECT shipment_id, hs_code, origin_port, dest_port, carrier FROM shipments WHERE status='Customs Hold' LIMIT 200"); rules = Q("SELECT * FROM customs_rules ORDER BY chapter"); rows = []
    for r in h.head(30).itertuples():
        ch, docs, rs, duty, goods = dom._required_docs(r.hs_code); miss = [d for d in docs if d not in dom._attached(r.shipment_id)]; rows.append({"shipment": r.shipment_id, "hs": r.hs_code, "goods": goods, "duty_pct": duty, "missing": ", ".join(miss), "route": f"{r.origin_port} → {r.dest_port}"})
    bl = [hero(c, "customs_watch", "Customs & tariff agent", "Requirements come from the customs_rules table by HS chapter. A hold lifts as soon as every required document is attached: the quote re-runs and the shipment clears."),
          B.kpis(("Shipments on hold", len(h), None, "Warning" if len(h) else None), ("Quotes on hold", int(Q("SELECT COUNT(*) n FROM freight_quotes WHERE status='On Hold'").n[0])), ("Rules in force", len(rules))),
          B.table(rows, ["shipment", "hs", "goods", B.num_col("duty_pct", "duty %", "num"), "missing", "route"], title="Held shipments and what each is missing · click a row", h=300, open={"col": "shipment"}, key="cu_h"), B.table(rules, list(rules.columns), title="Customs rules", h=230, key="cu_r")]
    if can(c, "resolve") or can(c, "quote"):
        held = Q("SELECT shipment_id ref FROM shipments WHERE status='Customs Hold' UNION SELECT quote_id FROM freight_quotes WHERE status='On Hold' LIMIT 200").ref.tolist(); alld = _docs_all()
        if held: bl.append(B.form("attach", [{"name": "ref", "label": "Shipment or quote on hold", "type": "select", "options": held, "value": held[0]}, {"name": "doc", "label": "Document", "type": "select", "options": alld, "value": alld[0]}], "Attach document", "/api/freight/attach", title="Attach a document", sub="The Customs agent re-checks on its next cycle; quotes re-run immediately."))
    if can(c, "view"): bl.append(ticket_block(c, "customs_watch", title="Open tickets from this agent", key="tk_cu"))
    return B.page("Customs", "Agent 6 · rules, holds, documents", bl, tabs, live=12)

PAGES = {"route": route, "quotes": quotes, "carriers": carriers, "weather": weather, "margin": margin, "customs": customs}


In [ ]:
%%writefile /content/freight_backend/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.utcnow() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.utcnow() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


In [ ]:
%%writefile /content/freight_backend/weather_context.py
import requests
from seed_data import BASE_PORTS

# Auto-built from BASE_PORTS in seed_data.py — always in sync with the port list
# (currently 50 ports). Each BASE_PORTS row is:
# (name, country, congestion_index, avg_dwell_days, lat, lon, region)
PORT_COORDS = {p[0]: (p[4], p[5]) for p in BASE_PORTS}

_DEFAULT_COORDS = (1.29, 103.85)  # Singapore Port as a safe fallback

def fetch_port_weather(port_name):
    coords = PORT_COORDS.get(port_name, _DEFAULT_COORDS)
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={coords[0]}&longitude={coords[1]}&current_weather=true"
        r = requests.get(url, timeout=3)
        if r.status_code == 200:
            cw = r.json().get('current_weather', {})
            windspeed = cw.get('windspeed', 15.0)
            severity = 1 if windspeed < 20 else (2 if windspeed < 40 else 3)
            return {'windspeed': windspeed, 'temperature': cw.get('temperature', 22.0), 'severity': severity}
    except Exception:
        pass
    return {'windspeed': 18.5, 'temperature': 24.0, 'severity': 2}


In [ ]:
%%writefile /content/freight_backend/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


In [ ]:
import base64
from pathlib import Path
asset_dir=Path('/content/freight_backend/static/vendor')
asset_dir.mkdir(parents=True, exist_ok=True)
world_topo_b64 = '...'
(asset_dir/'world.topo.json').write_bytes(base64.b64decode(world_topo_b64))
print('Sea-navigation coastline asset installed.')

In [ ]:
%%writefile /content/app.py
import os, re, hmac, hashlib, secrets, smtplib, sqlite3, time, datetime, html as _html
from contextlib import contextmanager
from email.message import EmailMessage

import bcrypt, jwt
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from streamlit_option_menu import option_menu

st.set_page_config(page_title="FreightQuote AI", page_icon="⚓", layout="wide", initial_sidebar_state="auto")

# ============================================================
# CONFIG  (every secret comes from environment variables that the
# notebook copies out of Colab Secrets -- nothing is hard-coded)
# ============================================================
JWT_SECRET = os.environ.get("JWT_SECRET")
ADMIN_USERNAME = os.environ.get("ADMIN_USERNAME")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD")
ADMIN_EMAIL = os.environ.get("ADMIN_EMAIL_ID") or (ADMIN_USERNAME if ADMIN_USERNAME and "@" in ADMIN_USERNAME else "admin@infosys.com")
DB_PATH = os.environ.get("DB_PATH", "infosys_portal.db")   # points to Google Drive when mounted
ON_DRIVE = DB_PATH.startswith("/content/drive")

APP_NAME = "FreightQuote AI"
PROJECT_TITLE = "Agentic AI for Maritime Freight Pricing and Route Optimization"
MAX_ATTEMPTS = 5        # wrong password / wrong OTP / wrong security answer attempts allowed
LOCK_SECONDS = 30       # how long the account stays locked
OTP_TTL = 300           # OTP validity in seconds
SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
    "What was the name of your first school?",
    "What is your favourite food?",
]

if not JWT_SECRET:
    st.error("JWT_SECRET is not configured. Run the setup cell (it loads your Colab Secrets) before launching the app.")
    st.stop()

# ============================================================
# STYLING
# ============================================================
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root{--ink:#0f172a;--muted:#64748b;--line:#e2e8f0;--brand:#0369a1;--brand2:#0891b2;--soft:#e0f2fe;}
html, body, .stApp, [data-testid="stAppViewContainer"]{font-family:'Inter',sans-serif !important;color:var(--ink);}
.stApp{background:radial-gradient(900px 520px at 6% -8%,#e0f2fe 0%,transparent 60%),radial-gradient(800px 500px at 100% 0%,#ccfbf1 0%,transparent 55%),#f8fafc !important;}
[data-testid="stHeader"]{background:transparent !important;}
footer,#MainMenu,[data-testid="stDecoration"],[data-testid="stAppDeployButton"],.stDeployButton{display:none !important;}
[data-testid="stMainBlockContainer"],.block-container{max-width:1180px !important;padding:2.2rem 2rem 2rem !important;}
h1,h2,h3,h4{font-family:'Inter',sans-serif !important;color:var(--ink) !important;letter-spacing:-.02em;}
[data-testid="stWidgetLabel"] p,label p{font-weight:600 !important;font-size:13px !important;color:#334155 !important;}

/* ---------- hero panel ---------- */
.hero{position:relative;overflow:hidden;border-radius:28px;padding:40px 40px 0;color:#fff;
  background:linear-gradient(160deg,rgba(14,116,144,.55) 0%,rgba(8,47,73,.88) 70%);border:1px solid rgba(255,255,255,.14);
  box-shadow:0 40px 80px -30px rgba(0,0,0,.65);}
.hero > *{position:relative;z-index:1;}
.brand{display:flex;align-items:center;gap:12px;font-weight:700;font-size:17px;color:#e0f2fe;}
.brand .logo{width:40px;height:40px;border-radius:12px;background:rgba(34,211,238,.16);border:1px solid rgba(34,211,238,.35);display:flex;align-items:center;justify-content:center;font-size:20px;}
.chip{display:inline-block;margin-top:30px;padding:5px 12px;border-radius:999px;font-size:11.5px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;color:#67e8f9;background:rgba(34,211,238,.12);border:1px solid rgba(34,211,238,.35);}
.hero h1{color:#fff !important;font-size:2.05rem !important;line-height:1.18;margin:14px 0 14px !important;font-weight:800;}
.hero h1 em{font-style:normal;background:linear-gradient(90deg,#67e8f9,#fcd34d);-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;}
.hero p.lead{color:#bae6fd;font-size:14.5px;line-height:1.65;margin:0;}
.feat{display:flex;gap:14px;align-items:flex-start;margin-top:18px;}
.feat .ic{flex:0 0 38px;height:38px;border-radius:10px;background:rgba(255,255,255,.10);border:1px solid rgba(255,255,255,.14);display:flex;align-items:center;justify-content:center;font-size:18px;}
.feat b{display:block;font-size:14px;color:#fff;}
.feat span{font-size:12.5px;color:#a5d8ef;}
.scene{display:block;width:calc(100% + 80px);margin:26px -40px 0;}

/* ---------- form card ---------- */
[data-testid="stForm"]{background:#fff;border:1px solid var(--line) !important;border-radius:24px !important;padding:34px 32px !important;box-shadow:0 25px 50px -22px rgba(15,23,42,.20);}
.card-title{font-size:26px;font-weight:800;margin:0;letter-spacing:-.02em;color:var(--ink);}
.card-sub{color:var(--muted);font-size:14px;margin:4px 0 20px;}

/* ---------- inputs ---------- */
div[data-baseweb="base-input"]{background:transparent !important;border:none !important;}
div[data-baseweb="input"],div[data-baseweb="select"] > div{background:#f8fafc !important;border:1.5px solid var(--line) !important;border-radius:12px !important;}
div[data-baseweb="input"]:focus-within,div[data-baseweb="select"] > div:focus-within{border-color:#0891b2 !important;box-shadow:0 0 0 4px rgba(8,145,178,.18) !important;background:#fff !important;}
input{color:var(--ink) !important;-webkit-text-fill-color:var(--ink) !important;font-size:14.5px !important;}
input::placeholder{color:#94a3b8 !important;-webkit-text-fill-color:#94a3b8 !important;}

/* ---------- buttons ---------- */
div[data-testid="stButton"] button,button[data-testid^="stBaseButton"]{border-radius:12px !important;height:46px;font-weight:700 !important;font-size:14px !important;transition:all .18s ease;}
button[data-testid="stBaseButton-primary"],button[data-testid="stBaseButton-primaryFormSubmit"],button[kind="primary"],button[kind="primaryFormSubmit"]{
  background:linear-gradient(135deg,#0c4a6e,#0891b2) !important;color:#fff !important;border:none !important;box-shadow:0 10px 20px -8px rgba(8,145,178,.65) !important;}
button[data-testid="stBaseButton-primary"]:hover,button[data-testid="stBaseButton-primaryFormSubmit"]:hover,button[kind="primary"]:hover,button[kind="primaryFormSubmit"]:hover{
  transform:translateY(-2px);box-shadow:0 14px 24px -8px rgba(8,145,178,.75) !important;}
button[data-testid="stBaseButton-secondary"],button[data-testid="stBaseButton-secondaryFormSubmit"],button[kind="secondary"],button[kind="secondaryFormSubmit"]{
  background:#fff !important;color:#0c4a6e !important;border:1.5px solid var(--line) !important;}
button[data-testid="stBaseButton-secondary"]:hover,button[data-testid="stBaseButton-secondaryFormSubmit"]:hover,button[kind="secondary"]:hover{
  border-color:#22d3ee !important;background:#ecfeff !important;}
[data-testid="stAlert"]{border-radius:12px !important;}

/* ---------- lock banner ---------- */
.lock{display:flex;align-items:center;gap:16px;border-radius:16px;padding:16px 18px;background:#fff1f2;border:1px solid #fecdd3;color:#9f1239;margin-top:12px;}
.lock .t{font-size:30px;font-weight:800;min-width:74px;text-align:center;background:#fff;border-radius:12px;padding:6px 8px;}
.lock b{display:block;font-size:15px;}
.lock span{font-size:13px;opacity:.9;}

/* ---------- dashboard ---------- */
section[data-testid="stSidebar"]{background:#fff !important;border-right:1px solid var(--line) !important;}
.banner{border-radius:20px;padding:26px 32px;display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:22px;
  background:linear-gradient(135deg,#04111d,#0c4a6e 60%,#0e7490);color:#fff;box-shadow:0 20px 40px -20px rgba(12,74,110,.7);}
.banner h1{color:#fff !important;margin:0 !important;font-size:1.55rem !important;font-weight:800;}
.banner .sub{color:#bae6fd;font-size:13.5px;margin-top:4px;}
.pill{background:rgba(255,255,255,.16);padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;white-space:nowrap;}
.kpi{background:#fff;border:1px solid var(--line);border-radius:18px;padding:20px;text-align:center;box-shadow:0 14px 28px -20px rgba(15,23,42,.25);}
.kpi .ic{font-size:26px;}
.kpi .v{font-size:26px;font-weight:800;color:var(--ink);margin-top:4px;}
.kpi .l{color:var(--muted);font-size:12px;font-weight:600;}
"""
st.markdown(f"<style>{CSS}</style>", unsafe_allow_html=True)

from urllib.parse import quote
_WAVE = ("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1440 220' preserveAspectRatio='none'>"
         "<path d='M0 90 C180 40 360 140 540 90 S900 40 1080 90 S1300 130 1440 80 V220 H0Z' fill='#06b6d4' fill-opacity='.10'/>"
         "<path d='M0 140 C200 100 380 190 600 140 S960 100 1160 150 S1360 170 1440 130 V220 H0Z' fill='#0891b2' fill-opacity='.16'/>"
         "<path d='M0 185 C240 160 420 215 700 185 S1100 160 1440 190 V220 H0Z' fill='#164e63' fill-opacity='.45'/></svg>")
_WAVE_URI = "data:image/svg+xml;utf8," + quote(_WAVE, safe="/:=;,'")
AUTH_BG_CSS = (".stApp{background:url(\"" + _WAVE_URI + "\") bottom/100% 240px no-repeat,"
               "radial-gradient(rgba(255,255,255,.055) 1px,transparent 1px) 0 0/30px 30px,"
               "radial-gradient(700px 420px at 88% 8%,rgba(6,182,212,.28),transparent 62%),"
               "radial-gradient(800px 520px at 0% 100%,rgba(14,116,144,.38),transparent 60%),"
               "linear-gradient(160deg,#030d17 0%,#07213a 55%,#0a3350 100%) !important;background-attachment:fixed !important;}"
               "[data-testid='stStatusWidget'],[data-testid='stStatusWidget'] *{color:#cbd5e1 !important;}")


def H(s):
    """Collapse indentation/newlines so markdown never turns HTML into a code block."""
    return " ".join(line.strip() for line in s.strip().splitlines())


def esc(s):
    return _html.escape(str(s), quote=True)


# ============================================================
# DATABASE  (SQLite file -- stored on Google Drive so it survives re-runs / restarts)
# ============================================================
@contextmanager
def db():
    conn = sqlite3.connect(DB_PATH, timeout=15)
    try:
        yield conn
        conn.commit()
        if ON_DRIVE:
            try:
                os.sync()          # push the change to Google Drive right away
            except Exception:
                pass
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


@st.cache_resource
def init_db():
    os.makedirs(os.path.dirname(os.path.abspath(DB_PATH)), exist_ok=True)
    with db() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
            password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
        cols = [r[1] for r in c.execute("PRAGMA table_info(users)")]
        if "pwd_fp" not in cols:                      # upgrade databases created by the old version
            c.execute("ALTER TABLE users ADD COLUMN pwd_fp TEXT")
        # Milestone 2 fields. Existing Milestone 1 accounts are retained and default to Customer.
        cols = [r[1] for r in c.execute("PRAGMA table_info(users)")]
        for col, ddl in [
            ("role", "ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'Customer'"),
            ("locked", "ALTER TABLE users ADD COLUMN locked INTEGER DEFAULT 0"),
            ("created_at", "ALTER TABLE users ADD COLUMN created_at TEXT"),
            ("last_login", "ALTER TABLE users ADD COLUMN last_login TEXT"),
            ("customer_id", "ALTER TABLE users ADD COLUMN customer_id TEXT"),
        ]:
            if col not in cols:
                c.execute(ddl)
        c.execute("UPDATE users SET role='Customer' WHERE role IS NULL OR role='' ")
        c.execute("UPDATE users SET locked=0 WHERE locked IS NULL")
        c.execute("UPDATE users SET created_at=CURRENT_TIMESTAMP WHERE created_at IS NULL")
        c.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_users_pwd_fp ON users(pwd_fp)")
        c.execute("""CREATE TABLE IF NOT EXISTS activity_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT DEFAULT CURRENT_TIMESTAMP,
            actor TEXT, action TEXT, target TEXT, detail TEXT)""")
        c.execute("""CREATE TABLE IF NOT EXISTS lockouts (
            key TEXT PRIMARY KEY, attempts INTEGER NOT NULL DEFAULT 0,
            locked_until REAL NOT NULL DEFAULT 0, last_try REAL NOT NULL DEFAULT 0)""")
        # the old version stored a hard-coded admin inside the users table -- remove it
        c.execute("DELETE FROM users WHERE email='infosys@ai'")
    return True


init_db()


@st.cache_resource
def dummy_hash():
    return bcrypt.hashpw(b"not-a-real-password", bcrypt.gensalt()).decode()


def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()


def check_txt(t, h):
    try:
        return bool(h) and bcrypt.checkpw(t.encode(), h.encode())
    except ValueError:        # e.g. password longer than bcrypt's 72-byte limit
        return False


def pwd_fp(p):
    """Keyed fingerprint of a password. bcrypt hashes are salted so they can't be compared
    between users; this lets us detect 'another account already uses this password'."""
    return hmac.new(("pw-fp|" + JWT_SECRET).encode(), p.encode(), hashlib.sha256).hexdigest()


def password_taken(p, exclude_email=None):
    with db() as c:
        if exclude_email:
            r = c.execute("SELECT 1 FROM users WHERE pwd_fp=? AND email<>?", (pwd_fp(p), exclude_email)).fetchone()
        else:
            r = c.execute("SELECT 1 FROM users WHERE pwd_fp=?", (pwd_fp(p),)).fetchone()
    return bool(r)


def find_user(ident):
    with db() as c:
        c.row_factory = sqlite3.Row
        r = c.execute("SELECT * FROM users WHERE lower(email)=lower(?) OR lower(username)=lower(?)",
                      (ident, ident)).fetchone()
        return dict(r) if r else None


def ensure_admin_account():
    """Create the platform admin once; keep its password bcrypt-hashed in the Milestone 1 database."""
    if not ADMIN_USERNAME or not ADMIN_PASSWORD:
        return
    email = (ADMIN_EMAIL or "admin@infosys.com").strip().lower()
    raw_name = ADMIN_USERNAME.strip()
    uname = raw_name.split("@", 1)[0] if "@" in raw_name else raw_name
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9 ._-]{2,29}", uname):
        uname = "platform_admin"
    user = find_user(email) or find_user(uname)
    stamp = datetime.datetime.now().isoformat(timespec="seconds")
    with db() as c:
        if user:
            c.execute("UPDATE users SET role='Admin', email=?, username=?, created_at=COALESCE(created_at, ?) WHERE id=?",
                      (email, uname, stamp, user["id"]))
            if not user.get("password_hash"):
                c.execute("UPDATE users SET password_hash=? WHERE id=?", (hash_txt(ADMIN_PASSWORD), user["id"]))
        else:
            c.execute("INSERT INTO users(username,email,password_hash,security_question,security_answer_hash,pwd_fp,role,locked,created_at) VALUES(?,?,?,?,?,?,?,?,?)",
                      (uname, email, hash_txt(ADMIN_PASSWORD), "", hash_txt("admin-recovery-disabled"), None, "Admin", 0, stamp))


ensure_admin_account()


# ============================================================
# VALIDATION
# ============================================================
# >=2 letters before @, >=2 letters between @ and the last dot, >=2 letters after the last dot
EMAIL_RE = re.compile(r"[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}[A-Za-z0-9-]*(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}")
USERNAME_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9 ._-]{2,29}")


def valid_email(e):
    return bool(EMAIL_RE.fullmatch(e.strip()))


def valid_username(u):
    return bool(USERNAME_RE.fullmatch(u.strip()))


def password_problems(p):
    out = []
    if len(p) < 8: out.append("at least 8 characters")
    if len(p) > 64: out.append("at most 64 characters")
    if not any(c.isupper() for c in p): out.append("an uppercase letter")
    if not any(c.islower() for c in p): out.append("a lowercase letter")
    if not any(c.isdigit() for c in p): out.append("a number")
    if not any(not c.isalnum() for c in p): out.append("a special symbol")
    return out


def pw_error(problems):
    return "⚠️ Password needs " + ", ".join(problems) + "."


# ============================================================
# BRUTE-FORCE LOCKOUT  (stored in the DB, so refreshing the page does NOT reset it)
# ============================================================
def lock_until(key):
    with db() as c:
        r = c.execute("SELECT locked_until FROM lockouts WHERE key=?", (key,)).fetchone()
    return r[0] if r else 0


def register_failure(key):
    """Count a failed attempt. Returns (attempts_so_far, locked_now)."""
    now = time.time()
    with db() as c:
        r = c.execute("SELECT attempts, last_try FROM lockouts WHERE key=?", (key,)).fetchone()
        attempts = (r[0] if r and now - r[1] < 600 else 0) + 1     # stale attempts (>10 min) are forgotten
        if attempts >= MAX_ATTEMPTS:
            c.execute("INSERT OR REPLACE INTO lockouts VALUES (?,?,?,?)", (key, 0, now + LOCK_SECONDS, now))
            if key.startswith(("login:", "admin:")):
                ident = key.split(":", 1)[1]
                c.execute("UPDATE users SET locked=1 WHERE lower(email)=lower(?) OR lower(username)=lower(?)", (ident, ident))
            return MAX_ATTEMPTS, True
        c.execute("INSERT OR REPLACE INTO lockouts VALUES (?,?,?,?)", (key, attempts, 0, now))
        return attempts, False


def clear_failures(key):
    with db() as c:
        c.execute("DELETE FROM lockouts WHERE key=?", (key,))


def lock_countdown(key, title="Too many failed attempts"):
    """Show a live 30-second countdown, then rerun so the form is usable again."""
    box = st.empty()
    until = lock_until(key)
    while time.time() < until:
        left = max(1, int(until - time.time() + 0.99))
        box.markdown(H(f"""<div class="lock"><div class="t">{left}s</div>
            <div><b>🔒 {esc(title)}</b><span>Account locked. You can try again when the timer reaches zero.</span></div></div>"""),
                     unsafe_allow_html=True)
        time.sleep(1)
    box.empty()
    st.rerun()


def guard(key, title="Too many failed attempts"):
    if lock_until(key) > time.time():
        lock_countdown(key, title)


def on_fail(key, msg, title="Too many failed attempts"):
    n, locked = register_failure(key)
    if locked:
        lock_countdown(key, title)
    st.error(f"{msg} {MAX_ATTEMPTS - n} attempt(s) left before a {LOCK_SECONDS}-second lock.")


# ============================================================
# JWT SESSIONS
# ============================================================
def make_jwt(sub, role):
    now = datetime.datetime.now(datetime.timezone.utc)
    return jwt.encode({"sub": sub, "role": role, "iat": now, "exp": now + datetime.timedelta(hours=2)},
                      JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


# ============================================================
# STATE / NAVIGATION
# ============================================================
DEFAULTS = {"token": None, "page": "Login", "flash": None, "reset_email": None, "reset_mode": None,
            "sq_text": None, "otp_code": None, "otp_expiry": None, "otp_verified": False}
for k, v in DEFAULTS.items():
    st.session_state.setdefault(k, v)


def navigate(p):
    st.session_state.page = p
    if p in {"Dashboard", "Copilot", "Route", "Quotes", "Carriers", "Weather", "Margin", "Customs", "Admin", "Profile"}:
        st.session_state["fq_nav"] = p
    st.rerun()


def reset_flow():
    for k in ("reset_email", "reset_mode", "sq_text", "otp_code", "otp_expiry"):
        st.session_state[k] = None
    st.session_state.otp_verified = False


def logout():
    st.session_state.token = None
    st.session_state.page = "Login"
    st.session_state["fq_nav"] = "Dashboard"
    st.rerun()


def mask_email(e):
    u, d = e.split("@", 1)
    return u[:2] + "•" * max(2, len(u) - 2) + "@" + d


# ============================================================
# OTP EMAIL
# ============================================================
def send_otp(email):
    sender, app_pw = os.environ.get("EMAIL_ADDRESS"), os.environ.get("EMAIL_PASSWORD")
    if not sender or not app_pw:
        return False, "Email service is not configured (EMAIL_ADDRESS / EMAIL_PASSWORD)."
    otp = f"{secrets.randbelow(10 ** 6):06d}"
    msg = EmailMessage()
    msg["Subject"] = f"{APP_NAME} - Password Reset OTP"
    msg["From"], msg["To"] = sender, email
    msg.set_content(f"Your password reset OTP is: {otp}\nIt expires in {OTP_TTL // 60} minutes. Do not share this code.")
    msg.add_alternative(H(f"""<div style="font-family:Arial,sans-serif;max-width:420px;margin:auto;border:1px solid #e2e8f0;border-radius:16px;padding:28px;text-align:center;">
        <div style="font-size:34px;">⚓</div><h2 style="margin:6px 0;color:#0f172a;">{APP_NAME}</h2>
        <p style="color:#64748b;">Use this code to reset your password</p>
        <div style="font-size:34px;font-weight:800;letter-spacing:8px;color:#0369a1;background:#e0f2fe;border-radius:12px;padding:14px;">{otp}</div>
        <p style="color:#64748b;font-size:13px;">Valid for {OTP_TTL // 60} minutes. If you didn't request this, ignore this email.</p></div>"""),
                          subtype="html")
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as s:
            s.login(sender, app_pw.replace(" ", ""))
            s.send_message(msg)
    except Exception:
        return False, "Could not send the OTP email. Check the Gmail App Password and try again."
    st.session_state.update(reset_email=email, reset_mode="otp", otp_code=otp,
                            otp_expiry=time.time() + OTP_TTL, otp_verified=False)
    return True, ""


# ============================================================
# UI PIECES
# ============================================================
FEATURES = [("💲", "Agentic freight pricing", "AI agents forecast rates and quote shipments."),
            ("🧭", "Route optimization", "Fuel- and time-efficient sea routes between ports."),
            ("🔐", "Secure access", "JWT sessions, OTP recovery and 30-second brute-force lock.")]

_SCENE = ("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 520 190'>"
          "<path d='M14 52 C120 -4 230 100 340 48 S470 14 506 58' fill='none' stroke='#67e8f9' stroke-width='2' stroke-dasharray='2 8' stroke-linecap='round'/>"
          "<circle cx='14' cy='52' r='6' fill='#fcd34d'/><circle cx='506' cy='58' r='6' fill='#34d399'/>"
          "<g transform='translate(150 70)'>"
          "<rect x='20' y='22' width='26' height='16' fill='#f97316'/><rect x='48' y='22' width='26' height='16' fill='#38bdf8'/>"
          "<rect x='76' y='22' width='26' height='16' fill='#facc15'/><rect x='104' y='22' width='26' height='16' fill='#f43f5e'/>"
          "<rect x='34' y='8' width='26' height='14' fill='#34d399'/><rect x='62' y='8' width='26' height='14' fill='#f97316'/>"
          "<rect x='90' y='8' width='26' height='14' fill='#38bdf8'/>"
          "<rect x='136' y='4' width='22' height='34' fill='#e2e8f0'/><rect x='140' y='11' width='14' height='6' fill='#0ea5e9'/>"
          "<rect x='145' y='-4' width='4' height='8' fill='#e2e8f0'/>"
          "<path d='M6 38 H172 L156 60 H24Z' fill='#0f172a'/><path d='M6 38 H172 L169 43 H9Z' fill='#ef4444'/></g>"
          "<path d='M0 132 C40 116 80 116 120 132 S200 148 240 132 S320 116 360 132 S440 148 480 132 S510 120 520 128 V190 H0Z' fill='#06b6d4' fill-opacity='.38'/>"
          "<path d='M0 152 C40 138 80 138 120 152 S200 166 240 152 S320 138 360 152 S440 166 480 152 S510 142 520 148 V190 H0Z' fill='#0e7490' fill-opacity='.65'/>"
          "<path d='M0 172 C60 162 120 162 180 172 S300 182 360 172 S470 162 520 172 V190 H0Z' fill='#082f49'/></svg>")
_SCENE_URI = "data:image/svg+xml;utf8," + quote(_SCENE, safe="/:=;,'")


def hero(lead):
    feats = "".join(f'<div class="feat"><div class="ic">{i}</div><div><b>{t}</b><span>{d}</span></div></div>'
                    for i, t, d in FEATURES)
    st.markdown(H(f"""<div class="hero"><div class="brand"><div class="logo">⚓</div>{APP_NAME}</div>
        <div class="chip">Infosys Springboard 7.0 · Milestone 1</div>
        <h1>Agentic AI for <em>Maritime Freight Pricing</em> and Route Optimization</h1>
        <p class="lead">{lead}</p>{feats}
        <img class="scene" src="{_SCENE_URI}" alt=""></div>"""), unsafe_allow_html=True)


def card_title(title, sub):
    st.markdown(H(f'<div class="card-title">{esc(title)}</div><div class="card-sub">{esc(sub)}</div>'),
                unsafe_allow_html=True)


def show_flash():
    if st.session_state.flash:
        st.success(st.session_state.flash)
        st.session_state.flash = None


def finish_reset(email, npw, cpw):
    """Shared last step of both recovery routes."""
    problems = password_problems(npw)
    if not npw or not cpw:
        st.error("⚠️ Please fill in both password fields.")
    elif problems:
        st.error(pw_error(problems))
    elif npw != cpw:
        st.error("❌ Passwords do not match.")
    elif password_taken(npw, exclude_email=email):
        st.error("❌ This password is not available. Please choose a different one.")
    else:
        try:
            with db() as c:
                c.execute("UPDATE users SET password_hash=?, pwd_fp=? WHERE email=?", (hash_txt(npw), pwd_fp(npw), email))
        except sqlite3.IntegrityError:
            st.error("❌ This password is not available. Please choose a different one.")
            return
        clear_failures("login:" + email)
        reset_flow()
        st.session_state.flash = "✅ Password updated. Please sign in with your new password."
        navigate("Login")



# ============================================================
# MILESTONE 2 — STREAMLIT RBAC + FREIGHTQUOTE BUSINESS LAYER
# ============================================================
ROLE_MENU = {
    "Admin": {"Dashboard", "Copilot", "Route", "Quotes", "Carriers", "Weather", "Margin", "Customs", "Admin"},
    "Operations Manager": {"Dashboard", "Copilot", "Route", "Carriers", "Weather", "Customs"},
    "Freight Broker": {"Dashboard", "Copilot", "Quotes", "Carriers", "Margin"},
    "Auditor": {"Dashboard", "Copilot", "Route", "Quotes", "Carriers", "Weather", "Margin", "Customs"},
    "Customer": {"Dashboard", "Copilot", "Quotes"},
}
ROLE_CAPS = {
    "Admin": {"users", "quotes_create", "agents_run", "storms_inject", "reroute", "customs_edit"},
    "Operations Manager": {"agents_run", "reroute"},
    "Freight Broker": {"quotes_create"},
    "Auditor": set(),
    "Customer": {"quotes_create_own"},
}
ROLE_MATRIX = [
    ("Admin", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓"),
    ("Operations Manager", "✓", "✓", "✗", "✓", "✓", "✗", "✓", "✓", "✗"),
    ("Freight Broker", "✓", "✗", "✓", "✓", "✗", "✓", "✗", "✗", "✗"),
    ("Auditor", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✗", "✗"),
    ("Customer", "✓", "✗", "✓", "✗", "✗", "✗", "✗", "✗", "✗"),
]

def can_page(role, page):
    return page == "Profile" or page in ROLE_MENU.get(role, set())

def can_do(role, capability):
    return role == "Admin" or capability in ROLE_CAPS.get(role, set())

def need(role, capability):
    if can_do(role, capability):
        return True
    st.error("403 — Your role does not have permission to perform this action.")
    return False

@st.cache_resource(show_spinner="Preparing FreightQuote data and agents…")
def load_freight_backend():
    import sys, importlib
    backend_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "freight_backend")
    if backend_path not in sys.path:
        sys.path.insert(0, backend_path)
    os.environ.setdefault("APP_KIND", "Freight")
    os.environ.setdefault("FREIGHTQUOTE_DATA_DIR", os.path.join(os.path.dirname(DB_PATH), "FreightQuoteData"))
    import db as FDB
    import seed_data as FSEED
    import live_engine as LE
    import domain_freight as DOM
    FDB.init_db()
    with FDB.get_conn() as c:
        have_ports = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
    if not have_ports:
        FSEED.seed_all()
    LE.init_live_tables()
    DOM.bootstrap()
    # The teacher-provided domain runtime runs in its own daemon worker (no FastAPI server).
    try:
        LE.start_runtime(DOM)
    except Exception:
        pass
    return {"db": FDB, "seed": FSEED, "LE": LE, "DOM": DOM}

def freight_query(B, sql, params=()):
    return B["DOM"].Q(sql, params)

def audit_admin(actor, action, target="", detail=""):
    with db() as c:
        c.execute("INSERT INTO activity_log(ts,actor,action,target,detail) VALUES(?,?,?,?,?)",
                  (datetime.datetime.now().isoformat(timespec="seconds"), actor, action, str(target), str(detail)[:500]))

def ensure_customer_link(user, B):
    """Give each Customer account a private FreightQuote customer_id."""
    if user.get("role") != "Customer":
        return user.get("customer_id")
    cid = user.get("customer_id") or f"CUST-U{int(user['id']):06d}"
    with B["db"].get_conn() as c:
        c.execute("INSERT OR IGNORE INTO customers(customer_id,name,industry,priority_tier,credit_risk) VALUES(?,?,?,?,?)",
                  (cid, user.get("username") or "Customer", "General", "Silver", 0.08))
    with db() as c:
        c.execute("UPDATE users SET customer_id=? WHERE id=?", (cid, user["id"]))
    return cid

def money(v):
    try: return f"${float(v):,.2f}"
    except Exception: return str(v)

def show_df(df, height=380):
    if df is None or getattr(df, "empty", True):
        st.info("No matching records found.")
    else:
        st.dataframe(df, use_container_width=True, hide_index=True, height=height)

def copilot_sql(question, role, customer_id, B):
    """Whitelisted, SQL-only Freight Copilot intents. Every answer is built from a DB query."""
    q = re.sub(r"\s+", " ", question.lower()).strip()
    # Common typing errors / spelling variants; the SQL and result are still database-backed.
    for wrong, right in {"shippments":"shipments", "shipement":"shipment", "carier":"carrier", "carrierss":"carriers", "congeston":"congestion", "custums":"customs", "quots":"quotes", "cheepest":"cheapest", "cheapestest":"cheapest"}.items():
        q = re.sub(r"\b" + re.escape(wrong) + r"\b", right, q)
    DOM = B["DOM"]
    # Greetings are conversational and intentionally do not query data.
    if re.search(r"^(hi|hello|hey|good morning|good afternoon|good evening)\b", q):
        return {"answer": "Hello! I am FreightQuote Copilot. I answer using the FreightQuote database. Try asking about shipments, carrier costs, congestion, quote margins or customs holds.", "sql": "-- Greeting; no database query required", "df": None, "rows": 0}
    # role -> data domain permissions. Customer queries always bind to the signed-in customer's ID.
    is_customer = role == "Customer"
    owner_filter = " WHERE customer_id=?" if is_customer else ""
    owner_params = (customer_id,) if is_customer else ()
    shipment_owner = " WHERE s.shipment_id IN (SELECT q.shipment_id FROM freight_quotes q WHERE q.customer_id=?)" if is_customer else ""
    if is_customer:
        allowed_ops = ("quote", "shipment")
    elif role == "Operations Manager":
        allowed_ops = ("shipment", "port", "carrier", "weather", "customs")
    elif role == "Freight Broker":
        allowed_ops = ("quote", "carrier", "margin")
    else:
        allowed_ops = ("quote", "shipment", "port", "carrier", "weather", "customs", "vessel")

    patterns = []
    def add(keywords, domain, sql, params=()):
        if domain not in allowed_ops: return
        if any(k in q for k in keywords):
            try:
                df = freight_query(B, sql, params)
                return {"answer": f"Database result for: **{question}**", "sql": sql, "df": df, "rows": len(df)}
            except Exception as e:
                return {"answer": f"The database query could not be completed: {e}", "sql": sql, "df": None, "rows": 0}
        return None

    if any(k in q for k in ["customs hold", "on customs hold", "held by customs"]):
        return add(["customs hold", "on customs hold", "held by customs"], "shipment", "SELECT shipment_id,origin_port,dest_port,carrier,status,eta,hs_code FROM shipments WHERE status='Customs Hold'" + (" AND shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else "") + " ORDER BY eta", (customer_id,) if is_customer else ())
    if any(k in q for k in ["cheapest carrier", "lowest cost carrier", "least expensive carrier"]) and " to " in q:
        lane = re.search(r"\bto\s+([a-z][a-z .'-]+)$", q)
        if lane:
            dest = "%" + lane.group(1).strip().title() + "%"
            sql = "SELECT q.carrier,MIN(q.final_price) AS lowest_quote,s.dest_port FROM freight_quotes q JOIN shipments s ON s.shipment_id=q.shipment_id WHERE s.dest_port LIKE ?" + (" AND q.customer_id=?" if is_customer else "") + " GROUP BY q.carrier,s.dest_port ORDER BY lowest_quote ASC LIMIT 10"
            return add(["cheapest carrier", "lowest cost carrier", "least expensive carrier"], "quote", sql, (dest, customer_id) if is_customer else (dest,))
    if any(k in q for k in ["cheapest carrier", "lowest cost carrier", "least expensive carrier"]):
        return add(["cheapest carrier", "lowest cost carrier", "least expensive carrier"], "carrier", "SELECT name AS carrier,rating,on_time_pct,avg_cost_index,risk_level FROM carriers ORDER BY avg_cost_index ASC LIMIT 10")
    if any(k in q for k in ["most congested", "highest congestion", "congested ports", "busy ports"]):
        return add(["most congested", "highest congestion", "congested ports", "busy ports"], "port", "SELECT port_name,country,congestion_index,avg_dwell_days FROM ports ORDER BY congestion_index DESC LIMIT 10")
    if any(k in q for k in ["least congested", "lowest congestion"]):
        return add(["least congested", "lowest congestion"], "port", "SELECT port_name,country,congestion_index,avg_dwell_days FROM ports ORDER BY congestion_index ASC LIMIT 10")
    if any(k in q for k in ["below 10% margin", "under 10% margin", "low margin", "quotes below 10"]):
        return add(["below 10% margin", "under 10% margin", "low margin", "quotes below 10"], "margin", "SELECT quote_id,shipment_id,carrier,final_price,margin_pct,status,created_at FROM freight_quotes WHERE margin_pct<10" + (" AND customer_id=?" if is_customer else "") + " ORDER BY margin_pct", (customer_id,) if is_customer else ())
    if any(k in q for k in ["highest margin", "best margin", "top margin"]):
        return add(["highest margin", "best margin", "top margin"], "margin", "SELECT quote_id,shipment_id,carrier,final_price,margin_pct,status FROM freight_quotes" + owner_filter + " ORDER BY margin_pct DESC LIMIT 10", owner_params)
    if any(k in q for k in ["pending approval", "awaiting approval"]):
        return add(["pending approval", "awaiting approval"], "quote", "SELECT quote_id,shipment_id,customer_id,carrier,final_price,margin_pct,status FROM freight_quotes WHERE status='Pending Approval'" + (" AND customer_id=?" if is_customer else "") + " ORDER BY created_at DESC", (customer_id,) if is_customer else ())
    if any(k in q for k in ["average quote", "average freight price", "average price"]):
        return add(["average quote", "average freight price", "average price"], "quote", "SELECT COUNT(*) AS quote_count,ROUND(AVG(final_price),2) AS average_price FROM freight_quotes" + owner_filter, owner_params)
    if any(k in q for k in ["how many quotes", "total quotes", "quote count", "number of quotes"]):
        return add(["how many quotes", "total quotes", "quote count", "number of quotes"], "quote", "SELECT COUNT(*) AS total_quotes FROM freight_quotes" + owner_filter, owner_params)
    if any(k in q for k in ["shipments by status", "shipment status", "status breakdown"]):
        return add(["shipments by status", "shipment status", "status breakdown"], "shipment", "SELECT status,COUNT(*) AS shipments FROM shipments" + (" WHERE shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else "") + " GROUP BY status ORDER BY shipments DESC", (customer_id,) if is_customer else ())
    if any(k in q for k in ["delayed shipment", "shipments delayed", "port congestion delay"]):
        return add(["delayed shipment", "shipments delayed", "port congestion delay"], "shipment", "SELECT shipment_id,origin_port,dest_port,carrier,status,eta,predicted_delay_risk FROM shipments WHERE status IN ('Port Congestion Delay','Anchorage Pending')" + (" AND shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else "") + " ORDER BY predicted_delay_risk DESC", (customer_id,) if is_customer else ())
    if any(k in q for k in ["on time carrier", "best carrier", "carrier reliability", "highest rated carrier"]):
        return add(["on time carrier", "best carrier", "carrier reliability", "highest rated carrier"], "carrier", "SELECT name AS carrier,rating,on_time_pct,avg_cost_index,risk_level FROM carriers ORDER BY on_time_pct DESC,rating DESC LIMIT 10")
    if "red sea" in q and any(k in q for k in ["storm", "weather", "risk"]):
        return add(["red sea"], "weather", "SELECT w.port_name,w.current_severity,w.forecast,w.wind_speed,w.wave_height,w.temperature FROM weather_risks w JOIN ports p ON p.port_name=w.port_name WHERE p.lat BETWEEN 12 AND 30 AND p.lon BETWEEN 32 AND 44 ORDER BY w.current_severity DESC,w.wind_speed DESC LIMIT 15")
    if any(k in q for k in ["weather risk", "high wind", "storm near", "storms near", "weather at ports"]):
        return add(["weather risk", "high wind", "storm near", "storms near", "weather at ports"], "weather", "SELECT port_name,current_severity,forecast,wind_speed,wave_height,temperature FROM weather_risks ORDER BY current_severity DESC,wind_speed DESC LIMIT 15")
    if any(k in q for k in ["track vessels", "vessels tracked", "where are vessels", "vessel positions"]):
        return add(["track vessels", "vessels tracked", "where are vessels", "vessel positions"], "vessel", "SELECT v.shipment_id,s.origin_port,s.dest_port,v.route_name,ROUND(v.progress*100,1) AS progress_pct,v.speed_kn,v.updated FROM vessels v LEFT JOIN shipments s ON s.shipment_id=v.shipment_id ORDER BY v.progress DESC LIMIT 25")
    if any(k in q for k in ["upcoming arrival", "arriving in 7 days", "next 7 days", "arrivals this week"]):
        return add(["upcoming arrival", "arriving in 7 days", "next 7 days", "arrivals this week"], "shipment", "SELECT shipment_id,origin_port,dest_port,carrier,status,eta FROM shipments WHERE eta BETWEEN date('now') AND date('now','+7 day')" + (" AND shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else "") + " ORDER BY eta", (customer_id,) if is_customer else ())
    if any(k in q for k in ["customs rules", "duty rate", "tariff rules", "required documents"]):
        return add(["customs rules", "duty rate", "tariff rules", "required documents"], "customs", "SELECT chapter,goods,duty_pct,docs,restricted FROM customs_rules ORDER BY chapter")
    if any(k in q for k in ["high risk carrier", "risky carrier", "carrier risk"]):
        return add(["high risk carrier", "risky carrier", "carrier risk"], "carrier", "SELECT name AS carrier,rating,on_time_pct,avg_cost_index,risk_level FROM carriers WHERE risk_level='High' ORDER BY avg_cost_index DESC")
    if any(k in q for k in ["total shipments", "how many shipments", "number of shipments"]):
        return add(["total shipments", "how many shipments", "number of shipments"], "shipment", "SELECT COUNT(*) AS total_shipments FROM shipments" + (" WHERE shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else ""), (customer_id,) if is_customer else ())
    if any(k in q for k in ["quotes to ", "shipments to "]):
        m = re.search(r"(?:quotes|shipments) to ([a-z][a-z .'-]+)", q)
        if m:
            dest = "%" + m.group(1).strip().title() + "%"
            if "quotes to " in q:
                return add(["quotes to "], "quote", "SELECT quote_id,shipment_id,carrier,final_price,margin_pct,status FROM freight_quotes WHERE shipment_id IN (SELECT shipment_id FROM shipments WHERE dest_port LIKE ?)" + (" AND customer_id=?" if is_customer else "") + " ORDER BY final_price", (dest, customer_id) if is_customer else (dest,))
            return add(["shipments to "], "shipment", "SELECT shipment_id,origin_port,dest_port,carrier,status,eta FROM shipments WHERE dest_port LIKE ?" + (" AND shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if is_customer else "") + " ORDER BY eta", (dest, customer_id) if is_customer else (dest,))
    # Generic analyst fallback is SQL-backed and only enabled for non-Customer roles.
    if not is_customer:
        try:
            import analyst
            result = analyst.answer("Freight", question, dom=DOM)
            if result:
                df = result.get("df")
                return {"answer": result.get("answer", "Query completed."), "sql": result.get("sql", ""), "df": df, "rows": len(df) if df is not None else 0}
        except Exception:
            pass
    return None


# ============================================================
# PAGES — NOT LOGGED IN
# ============================================================
if not st.session_state.token:
    # no sidebar on the auth pages
    st.markdown("<style>section[data-testid='stSidebar'],[data-testid='stSidebarCollapsedControl'],[data-testid='stExpandSidebarButton']{display:none !important;}</style>",
                unsafe_allow_html=True)
    st.markdown(f"<style>{AUTH_BG_CSS}</style>", unsafe_allow_html=True)
    if st.session_state.page not in ("Login", "Signup", "Forgot", "AdminLogin"):
        st.session_state.page = "Login"
    page = st.session_state.page

    left, right = st.columns([1.1, 1], gap="large", vertical_alignment="center")
    with left:
        if page == "Signup":
            hero("Pick a username, a strong unique password and a security question for account recovery.")
        elif page == "Forgot":
            hero("Recover access with your security question or a one-time code sent to your registered email.")
        elif page == "AdminLogin":
            hero("Restricted area. Sign in with the administrator credentials to view registered users.")
        else:
            hero("Sign in to quote freight, compare sea routes and track optimization results — protected by modern authentication.")

    with right:
        show_flash()

        # ---------------- LOGIN ----------------
        if page == "Login":
            with st.form("login_form"):
                card_title("Welcome back 👋", "Sign in to your account to continue")
                ident = st.text_input("Username or email", placeholder="username or you@example.com")
                pwd = st.text_input("Password", type="password", placeholder="Enter your password")
                submitted = st.form_submit_button("Sign in  →", type="primary", width="stretch")
            if submitted:
                ident = ident.strip()
                if not ident or not pwd:
                    st.error("⚠️ Username/email and password are required.")
                else:
                    user = find_user(ident)
                    key = "login:" + (user["email"] if user else ident.lower())
                    guard(key)
                    if user and int(user.get("locked") or 0):
                        st.error("This account is locked. Contact the administrator.")
                    else:
                        ok = check_txt(pwd, user["password_hash"] if user else dummy_hash()) and user is not None
                        if ok:
                            clear_failures(key)
                            if not user.get("pwd_fp"):          # backfill fingerprint for older accounts
                                try:
                                    with db() as c:
                                        c.execute("UPDATE users SET pwd_fp=? WHERE id=?", (pwd_fp(pwd), user["id"]))
                                except sqlite3.IntegrityError:
                                    pass
                            with db() as c:
                                c.execute("UPDATE users SET last_login=? WHERE id=?", (datetime.datetime.now().isoformat(timespec="seconds"), user["id"]))
                            st.session_state.token = make_jwt(user["email"], user.get("role") or "Customer")
                            navigate("Dashboard")
                        else:
                            on_fail(key, "❌ Invalid credentials.", "Too many failed sign-in attempts")
            c1, c2, c3 = st.columns(3)
            if c1.button("Create account", width="stretch"): navigate("Signup")
            if c2.button("Forgot password?", width="stretch"): navigate("Forgot")
            if c3.button("🛡️ Admin login", width="stretch"): navigate("AdminLogin")

        # ---------------- SIGNUP ----------------
        elif page == "Signup":
            with st.form("signup_form"):
                card_title("Create account", f"Join {APP_NAME} — all fields are required")
                uname = st.text_input("Username", placeholder="e.g. jane.doe")
                email = st.text_input("Email address", placeholder="you@example.com")
                p1, p2 = st.columns(2)
                pwd = p1.text_input("Password", type="password", placeholder="Create a password")
                cpw = p2.text_input("Confirm password", type="password", placeholder="Re-enter password")
                st.caption("Password: 8+ characters with an uppercase letter, lowercase letter, number and special symbol. "
                           "It must be different from every other user's password.")
                sq = st.selectbox("Security question", SECURITY_QUESTIONS)
                sa = st.text_input("Security answer", placeholder="Your answer")
                submitted = st.form_submit_button("Create account  →", type="primary", width="stretch")
            if submitted:
                uname, email, sa = uname.strip(), email.strip().lower(), sa.strip()
                problems = password_problems(pwd)
                if not all([uname, email, pwd, cpw, sa]):
                    st.error("⚠️ Please fill in all required fields.")
                elif not valid_username(uname):
                    st.error("❌ Username must be 3–30 characters (letters, numbers, space, . _ -).")
                elif not valid_email(email):
                    st.error("❌ Enter a valid email address (e.g. ab@cd.ef).")
                elif problems:
                    st.error(pw_error(problems))
                elif pwd != cpw:
                    st.error("❌ Passwords do not match.")
                else:
                    with db() as c:
                        name_taken = c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?)", (uname,)).fetchone()
                        mail_taken = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                    if name_taken:
                        st.error("❌ That username is already taken. Please choose another.")
                    elif mail_taken:
                        st.error("❌ That email is already registered.")
                    elif password_taken(pwd):
                        st.error("❌ This password is not available. Please choose a different one.")
                    else:
                        try:
                            with db() as c:
                                c.execute("INSERT INTO users (username,email,password_hash,security_question,security_answer_hash,pwd_fp,role,locked,created_at) VALUES (?,?,?,?,?,?,?,?,?)",
                                          (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower()), pwd_fp(pwd), "Customer", 0, datetime.datetime.now().isoformat(timespec="seconds")))
                        except sqlite3.IntegrityError:
                            st.error("❌ Username, email or password is not available. Please change it and try again.")
                        else:
                            st.session_state.flash = "✅ Account created! Please sign in."
                            navigate("Login")
            if st.button("← Back to sign in", width="stretch"): navigate("Login")

        # ---------------- FORGOT PASSWORD ----------------
        elif page == "Forgot":
            mode, remail = st.session_state.reset_mode, st.session_state.reset_email

            if not mode:   # step 1: pick the route
                with st.form("forgot_start"):
                    card_title("Reset password", "Verify it's you with a security question or an email OTP")
                    ident = st.text_input("Username or registered email", placeholder="username or you@example.com")
                    b1, b2 = st.columns(2)
                    sq_btn = b1.form_submit_button("Security question", type="secondary", width="stretch")
                    otp_btn = b2.form_submit_button("Email OTP", type="primary", width="stretch")
                if sq_btn or otp_btn:
                    user = find_user(ident.strip()) if ident.strip() else None
                    if not ident.strip():
                        st.error("⚠️ Enter your username or registered email.")
                    elif not user:
                        st.error("❌ No account found with that username or email.")
                    elif sq_btn:
                        st.session_state.update(reset_email=user["email"], reset_mode="sq", sq_text=user["security_question"])
                        st.rerun()
                    else:
                        guard("otp:" + user["email"], "Too many wrong OTP attempts")
                        ok, err = send_otp(user["email"])
                        if ok: st.rerun()
                        else: st.error("❌ " + err)

            elif mode == "sq":   # route A: security question
                key = "sq:" + remail
                with st.form("forgot_sq"):
                    card_title("Security question", "Answer correctly, then choose a new password")
                    st.info(f"❓ {st.session_state.sq_text}")
                    ans = st.text_input("Your answer")
                    n1, n2 = st.columns(2)
                    npw = n1.text_input("New password", type="password")
                    cpw = n2.text_input("Confirm new password", type="password")
                    go_btn = st.form_submit_button("Reset password  →", type="primary", width="stretch")
                if go_btn:
                    if not ans.strip() or not npw or not cpw:
                        st.error("⚠️ All fields are required.")
                    else:
                        guard(key, "Too many wrong answers")
                        with db() as c:
                            r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (remail,)).fetchone()
                        if r and check_txt(ans.strip().lower(), r[0]):
                            clear_failures(key)
                            finish_reset(remail, npw, cpw)
                        else:
                            on_fail(key, "❌ Incorrect security answer.", "Too many wrong answers")

            elif mode == "otp" and not st.session_state.otp_verified:   # route B, step 1: verify OTP
                key = "otp:" + remail
                with st.form("forgot_otp"):
                    card_title("Enter OTP", f"We emailed a 6-digit code to {mask_email(remail)}")
                    entered = st.text_input("6-digit OTP", max_chars=6, placeholder="••••••")
                    verify = st.form_submit_button("Verify OTP  →", type="primary", width="stretch")
                if verify:
                    guard(key, "Too many wrong OTP attempts")
                    if not st.session_state.otp_code:
                        st.error("⚠️ No active OTP. Click “Resend OTP” to get a new one.")
                    elif time.time() > st.session_state.otp_expiry:
                        st.session_state.otp_code = None
                        st.error("⌛ OTP expired. Click “Resend OTP” to get a new one.")
                    elif not entered.strip():
                        st.error("⚠️ Enter the OTP.")
                    elif secrets.compare_digest(entered.strip().encode(), st.session_state.otp_code.encode()):
                        clear_failures(key)
                        st.session_state.otp_verified = True
                        st.rerun()
                    else:
                        n, locked = register_failure(key)
                        if locked:
                            st.session_state.otp_code = None     # the old code is dead; a new one is needed
                            lock_countdown(key, "Too many wrong OTP attempts")
                        st.error(f"❌ Incorrect OTP. {MAX_ATTEMPTS - n} attempt(s) left before a {LOCK_SECONDS}-second lock.")
                if st.button("↻ Resend OTP", width="stretch"):
                    guard(key, "Too many wrong OTP attempts")
                    ok, err = send_otp(remail)
                    if ok:
                        st.success("📧 A new OTP has been sent.")
                    else:
                        st.error("❌ " + err)
                if not st.session_state.otp_code:
                    st.caption("The previous OTP is no longer valid — request a new one.")

            elif mode == "otp":   # route B, step 2: new password
                with st.form("forgot_newpw"):
                    card_title("Set a new password", "OTP verified ✅ — choose a strong password")
                    n1, n2 = st.columns(2)
                    npw = n1.text_input("New password", type="password")
                    cpw = n2.text_input("Confirm new password", type="password")
                    go_btn = st.form_submit_button("Reset password  →", type="primary", width="stretch")
                if go_btn:
                    if time.time() > (st.session_state.otp_expiry or 0):
                        st.session_state.otp_verified = False
                        st.session_state.otp_code = None
                        st.error("⌛ OTP expired. Please start again.")
                    else:
                        finish_reset(remail, npw, cpw)

            if st.button("← Cancel and go back", width="stretch"):
                reset_flow()
                navigate("Login")

        # ---------------- ADMIN LOGIN ----------------
        elif page == "AdminLogin":
            with st.form("admin_form"):
                card_title("Admin sign in 🛡️", "Restricted to administrators")
                a_user = st.text_input("Admin username")
                a_pass = st.text_input("Admin password", type="password")
                submitted = st.form_submit_button("Sign in as admin  →", type="primary", width="stretch")
            if submitted:
                if not a_user.strip() or not a_pass:
                    st.error("⚠️ Admin username and password are required.")
                elif not ADMIN_USERNAME or not ADMIN_PASSWORD:
                    st.error("Admin credentials are not configured (ADMIN_USERNAME / ADMIN_PASSWORD).")
                else:
                    key = "admin:" + a_user.strip().lower()
                    guard(key, "Too many failed admin attempts")
                    admin_user = find_user(ADMIN_EMAIL)
                    names = {str(ADMIN_USERNAME).lower(), str(ADMIN_EMAIL).lower()}
                    if admin_user:
                        names.add(str(admin_user.get("username", "")).lower())
                    ok_u = a_user.strip().lower() in names
                    ok_p = bool(admin_user) and check_txt(a_pass, admin_user.get("password_hash"))
                    if ok_u and ok_p and not int(admin_user.get("locked") or 0):
                        clear_failures(key)
                        with db() as c:
                            c.execute("UPDATE users SET last_login=? WHERE id=?", (datetime.datetime.now().isoformat(timespec="seconds"), admin_user["id"]))
                        st.session_state.token = make_jwt(admin_user["email"], "Admin")
                        navigate("Dashboard")
                    else:
                        on_fail(key, "❌ Invalid admin credentials.", "Too many failed admin attempts")
            if st.button("← Back to user sign in", width="stretch"): navigate("Login")

# ============================================================
# PAGES — LOGGED IN (MILESTONE 2 STREAMLIT APPLICATION)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        logout()
    user = find_user(payload.get("sub", ""))
    if not user or int(user.get("locked") or 0):
        st.session_state.token = None
        st.session_state.page = "Login"
        st.error("Your account is locked or no longer exists. Please sign in again or contact the administrator.")
        st.stop()
    role = user.get("role") or "Customer"  # database role is authoritative; a role change takes effect immediately
    if payload.get("role") != role:
        st.session_state.token = make_jwt(user["email"], role)
    username, email = user["username"], user["email"]

    try:
        B = load_freight_backend()
        customer_id = ensure_customer_link(user, B) if role == "Customer" else user.get("customer_id")
        DOM, LE = B["DOM"], B["LE"]
    except Exception as e:
        st.error("FreightQuote could not initialize its local data engine.")
        st.exception(e)
        st.stop()

    pages = [p for p in ["Dashboard", "Copilot", "Route", "Quotes", "Carriers", "Weather", "Margin", "Customs", "Admin", "Profile"] if can_page(role, p)]
    with st.sidebar:
        st.markdown("<div style='text-align:center;padding:12px 4px 16px'><div style='font-size:34px'>⚓</div><div style='font-size:18px;font-weight:800'>FreightQuote AI</div><div style='font-size:12px;color:#64748b'>Milestone 1 + Milestone 2 · Streamlit</div></div>", unsafe_allow_html=True)
        st.caption(f"Signed in as **{username}**")
        st.caption(f"Role: **{role}**")
        page = st.radio("Navigation", pages, key="fq_nav")
        st.divider()
        dark_mode = st.toggle("Dark mode", key="fq_dark", value=False)
        if st.button("Sign out", use_container_width=True):
            logout()
    if dark_mode:
        st.markdown("<style>.stApp{background:#0b1220!important;color:#e5e7eb!important} section[data-testid='stSidebar']{background:#111827!important} h1,h2,h3,p,label{color:#e5e7eb!important} [data-testid='stDataFrame'],[data-testid='stForm']{filter:brightness(.92)}</style>", unsafe_allow_html=True)
    if not can_page(role, page):
        st.error("403 — This page is not available for your role.")
        st.stop()

    st.markdown(f"<div class='banner'><div><h1>FreightQuote AI</h1><div class='sub'>Agentic AI for maritime freight pricing, routing and operations</div></div><div class='pill'>{esc(role)} · {esc(username)}</div></div>", unsafe_allow_html=True)

    def metric_row(items):
        cols = st.columns(len(items))
        for col, (label, val, icon) in zip(cols, items):
            col.markdown(f"<div class='kpi'><div class='ic'>{icon}</div><div class='v'>{esc(val)}</div><div class='l'>{esc(label)}</div></div>", unsafe_allow_html=True)

    def open_actions():
        return freight_query(B, "SELECT action_id,ts,type,target,severity,status,detail FROM agent_actions WHERE status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 50")

    # ---------------- DASHBOARD ----------------
    if page == "Dashboard":
        ship_sql = "SELECT COUNT(*) n FROM shipments" + (" WHERE shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if role=="Customer" else "")
        shipments = int(freight_query(B, ship_sql, (customer_id,) if role=="Customer" else ()).iloc[0]["n"])
        quotes = int(freight_query(B, "SELECT COUNT(*) n FROM freight_quotes" + (" WHERE customer_id=?" if role=="Customer" else ""), (customer_id,) if role=="Customer" else ()).iloc[0]["n"])
        vessels = int(freight_query(B, "SELECT COUNT(*) n FROM vessels" + (" WHERE shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if role=="Customer" else ""), (customer_id,) if role=="Customer" else ()).iloc[0]["n"])
        holds = int(freight_query(B, "SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'" + (" AND shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if role=="Customer" else ""), (customer_id,) if role=="Customer" else ()).iloc[0]["n"])
        metric_row([("Shipments", f"{shipments:,}", "🚢"), ("Your quotes" if role=="Customer" else "Freight quotes", f"{quotes:,}", "💲"), ("Vessels tracked", f"{vessels:,}", "🧭"), ("Customs holds", f"{holds:,}", "📦")])
        left, right = st.columns(2)
        with left:
            st.subheader("Shipment status")
            sql = "SELECT status,COUNT(*) AS shipments FROM shipments" + (" WHERE shipment_id IN (SELECT shipment_id FROM freight_quotes WHERE customer_id=?)" if role=="Customer" else "") + " GROUP BY status ORDER BY shipments DESC"
            df = freight_query(B, sql, (customer_id,) if role=="Customer" else ())
            if not df.empty: st.plotly_chart(go.Figure(go.Pie(labels=df.status, values=df.shipments, hole=.58)), use_container_width=True)
        with right:
            st.subheader("Carrier performance")
            if role not in ("Customer",):
                df = freight_query(B, "SELECT name,on_time_pct FROM carriers ORDER BY on_time_pct DESC LIMIT 10")
                if not df.empty: st.plotly_chart(go.Figure(go.Bar(x=df.on_time_pct, y=df.name, orientation="h")), use_container_width=True)
            else:
                st.info("Use Quotes to create and monitor your own freight requests.")
        st.subheader("Recent shipments")
        sql = "SELECT s.shipment_id,s.origin_port,s.dest_port,s.carrier,s.status,s.eta,s.predicted_delay_risk FROM shipments s"
        if role == "Customer": sql += " JOIN freight_quotes q ON q.shipment_id=s.shipment_id WHERE q.customer_id=?"
        sql += " ORDER BY s.eta LIMIT 20"
        show_df(freight_query(B, sql, (customer_id,) if role=="Customer" else ()))
        if can_do(role, "agents_run"):
            st.subheader("Agent control")
            if st.button("▶ Run agent cycle now", type="primary") and need(role, "agents_run"):
                with st.spinner("Running the monitoring agents…"):
                    result = LE.run_cycle(DOM, trigger="manual", request=f"Run requested by {username}")
                st.success("Agent cycle completed.")
                st.write(result.get("narrative", "Cycle finished."))
                st.json(result.get("agents", {}))
        else:
            with st.expander("Permission check — restricted agent action"):
                if st.button("Try to run agents (403 test)"):
                    need(role, "agents_run")

    # ---------------- COPILOT ----------------
    elif page == "Copilot":
        st.subheader("AI Copilot · SQL-backed answers")
        st.caption("Ask about database records. Answers show the SQL query and number of returned rows.")
        prompt_examples = ["Which carrier is cheapest?", "How many shipments are on customs hold?", "Show quotes below 10% margin", "Which ports are most congested?"]
        c = st.columns(4)
        for i, example in enumerate(prompt_examples):
            if c[i].button(example, use_container_width=True, key=f"copilot_example_{i}"):
                st.session_state["fq_prompt"] = example
        for msg in B["db"].load_chat_history(username=email, limit=30):
            with st.chat_message("assistant" if msg["role"]=="assistant" else "user"):
                st.markdown(msg["content"])
        question = st.chat_input("Ask FreightQuote Copilot…")
        preset = st.session_state.pop("fq_prompt", None)
        question = question or preset
        if question:
            B["db"].save_chat_message(email, "user", question)
            with st.chat_message("user"): st.markdown(question)
            result = copilot_sql(question, role, customer_id, B)
            with st.chat_message("assistant"):
                if result is None:
                    valid = ("How many shipments are on customs hold?", "Which carrier is cheapest?", "Which ports are most congested?")
                    answer = "I can answer questions that are covered by your role and the FreightQuote database. Try one of these:\n" + "\n".join(f"- {x}" for x in valid)
                    st.markdown(answer)
                    B["db"].save_chat_message(email, "assistant", answer)
                else:
                    st.markdown(result["answer"])
                    st.caption(f"Rows returned: {result['rows']}")
                    with st.expander("SQL used"):
                        st.code(result["sql"], language="sql")
                    if result.get("df") is not None: show_df(result["df"])
                    B["db"].save_chat_message(email, "assistant", result["answer"])

    # ---------------- ROUTE AI ----------------
    elif page == "Route":
        st.subheader("Route AI · sea-lane planning")
        ports = freight_query(B, "SELECT port_name,lat,lon,congestion_index,avg_dwell_days FROM ports ORDER BY port_name")
        names = ports.port_name.tolist()
        if len(names) >= 2:
            x, y = st.columns(2)
            origin = x.selectbox("Origin port", names, index=0)
            dest = y.selectbox("Destination port", names, index=min(1,len(names)-1))
            if st.button("Calculate sea route", type="primary"):
                if origin == dest: st.warning("Choose different origin and destination ports.")
                else:
                    po, pdest = DOM.port_xy(origin), DOM.port_xy(dest)
                    with st.spinner("Calculating water-only route options…"):
                        opts = DOM.route_options(po[:2], pdest[:2])
                    for i, opt in enumerate(opts[:4]):
                        st.markdown(f"**Option {i+1}: {opt['name']}** · {opt['nm']:,} nautical miles")
                        st.caption("Via: " + (", ".join(opt.get("via", [])) or "Direct sea corridor"))
                        if opt.get("chain"):
                            try: st.map(pd.DataFrame(opt["chain"], columns=["lat","lon"]))
                            except Exception: pass
        st.subheader("Port congestion and dwell")
        show_df(freight_query(B, "SELECT port_name,country,congestion_index,avg_dwell_days FROM ports ORDER BY congestion_index DESC LIMIT 25"))
        if can_do(role,"agents_run") and st.button("Run route and fleet agents") and need(role,"agents_run"):
            r=LE.run_cycle(DOM,trigger="route_page"); st.success(r.get("narrative","Agents completed."))
        if can_do(role,"reroute"):
            vessels_df=freight_query(B,"SELECT shipment_id,route_name,progress,lat,lon FROM vessels ORDER BY shipment_id LIMIT 300")
            if not vessels_df.empty:
                sid=st.selectbox("Vessel to re-route",vessels_df.shipment_id.tolist())
                if st.button("Re-route around active storms") and need(role,"reroute"):
                    act={"type":"reroute","target":sid}
                    with LE._DB_LOCK, LE.conn() as con: msg=DOM.apply(act,con)
                    st.success(msg or "No re-route was needed.")
        else:
            with st.expander("Permission check — restricted re-route action"):
                if st.button("Try to re-route (403 test)"):
                    need(role,"reroute")

    # ---------------- QUOTES ----------------
    elif page == "Quotes":
        st.subheader("Spot Quotes · seven-step quote pipeline")
        quote_cap = "quotes_create" if role!="Customer" else "quotes_create_own"
        if can_do(role, quote_cap):
            ports = freight_query(B,"SELECT port_name FROM ports ORDER BY port_name").port_name.tolist()
            customers = freight_query(B,"SELECT customer_id,name FROM customers ORDER BY name")
            with st.form("new_quote_form"):
                if role == "Customer":
                    st.text_input("Customer account", value=f"{username} ({customer_id})", disabled=True)
                    quote_customer=customer_id
                else:
                    customer_opts={f"{r.customer_id} · {r.name}":r.customer_id for r in customers.itertuples()}
                    customer_label=st.selectbox("Customer",list(customer_opts))
                    quote_customer=customer_opts[customer_label]
                a,b=st.columns(2)
                origin=a.selectbox("Origin",ports,index=0)
                dest=b.selectbox("Destination",ports,index=min(1,len(ports)-1))
                cargo=st.selectbox("Cargo type",list(DOM.CARGO.keys()))
                weight=st.number_input("Weight (tonnes)",min_value=0.1,max_value=5000.0,value=20.0,step=0.5)
                hs=st.text_input("HS code",value="HS-8517")
                docs=st.multiselect("Documents already provided",["Commercial Invoice","Packing List","Bill of Lading","Certificate of Origin","BIS Certificate","Import License","Drug Control Certificate","IMDG Certificate"],default=["Commercial Invoice","Packing List","Bill of Lading"])
                submitted=st.form_submit_button("Generate freight quote",type="primary",use_container_width=True)
            if submitted and need(role, quote_cap):
                if origin==dest: st.error("Origin and destination must be different.")
                elif not re.search(r"\d{2}",hs): st.error("Enter a valid HS code with at least two digits.")
                else:
                    with st.spinner("Running routing, weather, pricing, customs, margin and document agents…"):
                        rid=DOM.submit_request(origin,dest,cargo,float(weight),hs,quote_customer,docs_provided=docs)
                        result=DOM.run_quote(rid)
                    if result:
                        st.success("Quote generated successfully.")
                        st.write(result.get("narrative","Quote created."))
                        st.json(result)
                    else: st.error("Quote pipeline could not complete. Check the request status and agent logs.")
        else:
            st.info("Read-only access: your Auditor role can view quotes but cannot create them.")
        st.subheader("Quote history")
        if role == "Customer":
            sql="SELECT quote_id,shipment_id,carrier,final_price,status,created_at FROM freight_quotes"
        else:
            sql="SELECT quote_id,shipment_id,customer_id,carrier,base_cost,insurance,customs_fee,fuel_surcharge,final_price,margin_pct,status,created_at FROM freight_quotes"
        params=()
        if role=="Customer": sql += " WHERE customer_id=?"; params=(customer_id,)
        sql += " ORDER BY created_at DESC LIMIT 100"
        qdf=freight_query(B,sql,params)
        show_df(qdf)
        if not qdf.empty:
            st.download_button("Download quote history CSV",qdf.to_csv(index=False).encode(),"freight_quotes.csv","text/csv")

    # ---------------- CARRIERS ----------------
    elif page == "Carriers":
        st.subheader("Carrier intelligence")
        show_df(freight_query(B,"SELECT carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level FROM carriers ORDER BY rating DESC,on_time_pct DESC"),500)
        st.caption("Carrier ratings, on-time percentage, cost index and risk level are read from the FreightQuote database.")

    # ---------------- WEATHER ----------------
    elif page == "Weather":
        st.subheader("Weather risk · ports and active storms")
        show_df(freight_query(B,"SELECT port_name,current_severity,forecast,wind_speed,wave_height,temperature FROM weather_risks ORDER BY current_severity DESC,wind_speed DESC"),500)
        storms=DOM.storms()
        st.write(f"Active storm cells: **{len(storms)}**")
        if storms: st.json(storms)
        if can_do(role,"storms_inject"):
            st.markdown("#### Storm scenario test")
            plist=freight_query(B,"SELECT port_name,lat,lon FROM ports ORDER BY port_name")
            pnames=plist.port_name.tolist()
            selected=st.selectbox("Port near which to inject a test storm",pnames)
            wind=st.slider("Wind speed (m/s)",10.0,60.0,28.0,1.0)
            if st.button("Inject storm",type="primary") and need(role,"storms_inject"):
                pr=plist[plist.port_name==selected].iloc[0]
                DOM.inject_storm(float(pr.lat),float(pr.lon),float(wind))
                st.success(f"Test storm injected near {selected}.")
        if can_do(role,"agents_run") and st.button("Run weather-risk agents") and need(role,"agents_run"):
            r=LE.run_cycle(DOM,trigger="weather_page"); st.success(r.get("narrative","Weather agents completed."))

    # ---------------- MARGIN ----------------
    elif page == "Margin":
        st.subheader("Margin intelligence")
        sql="SELECT quote_id,customer_id,carrier,base_cost,final_price,margin_pct,status,created_at FROM freight_quotes"
        params=()
        if role=="Customer": sql+=" WHERE customer_id=?";params=(customer_id,)
        qdf=freight_query(B,sql+" ORDER BY margin_pct DESC",params)
        show_df(qdf)
        if not qdf.empty:
            st.plotly_chart(go.Figure(go.Histogram(x=qdf.margin_pct,nbinsx=18)),use_container_width=True)
            st.metric("Average margin",f"{qdf.margin_pct.mean():.1f}%")

    # ---------------- CUSTOMS ----------------
    elif page == "Customs":
        st.subheader("Customs rules, holds and documents")
        st.markdown("**Rules by HS chapter**")
        show_df(freight_query(B,"SELECT chapter,goods,duty_pct,docs,restricted FROM customs_rules ORDER BY chapter"))
        st.markdown("**Shipments requiring customs attention**")
        held=freight_query(B,"SELECT shipment_id,origin_port,dest_port,carrier,hs_code,status,eta FROM shipments WHERE status='Customs Hold' ORDER BY eta")
        show_df(held)
        if can_do(role,"customs_edit"):
            st.markdown("#### Attach a missing customs document")
            if not held.empty:
                sid=st.selectbox("Shipment",held.shipment_id.tolist())
                doc=st.selectbox("Document",["Commercial Invoice","Packing List","Bill of Lading","Certificate of Origin","BIS Certificate","Import License","Drug Control Certificate","IMDG Certificate"])
                if st.button("Attach document") and need(role,"customs_edit"):
                    DOM.attach_doc(sid,doc,by=username); st.success("Document attached.")

    # ---------------- PROFILE ----------------
    elif page == "Profile":
        st.subheader("My Profile")
        joined=user.get("created_at") or "—"; last=user.get("last_login") or "—"
        metric_row([("Username",username,"👤"),("Role",role,"🛡️"),("Joined",str(joined)[:10],"📅"),("Last login",str(last)[:16],"🕒")])
        st.write("**Email:**",email)
        st.caption("Your password hash is never displayed.")
        with st.form("profile_username_form"):
            new_username=st.text_input("Change username",value=username)
            save_name=st.form_submit_button("Update username")
        if save_name:
            if not valid_username(new_username): st.error("Username must be 3–30 characters (letters, numbers, spaces, . _ -).")
            else:
                with db() as c: clash=c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?) AND id<>?",(new_username,user["id"])).fetchone()
                if clash: st.error("That username is already in use.")
                else:
                    with db() as c: c.execute("UPDATE users SET username=? WHERE id=?",(new_username,user["id"]))
                    st.success("Username updated."); st.rerun()
        with st.form("profile_password_form"):
            current_pw=st.text_input("Current password",type="password")
            new_pw=st.text_input("New password",type="password")
            confirm_pw=st.text_input("Confirm new password",type="password")
            save_pw=st.form_submit_button("Change password")
        if save_pw:
            if not check_txt(current_pw,user.get("password_hash")): st.error("Current password is incorrect.")
            elif password_problems(new_pw): st.error(pw_error(password_problems(new_pw)))
            elif new_pw!=confirm_pw: st.error("New passwords do not match.")
            elif password_taken(new_pw,exclude_email=email): st.error("This password is already used by another account.")
            else:
                with db() as c: c.execute("UPDATE users SET password_hash=?,pwd_fp=? WHERE id=?",(hash_txt(new_pw),pwd_fp(new_pw),user["id"]))
                st.success("Password changed successfully. Please sign in again."); logout()

    # ---------------- ADMIN DASHBOARD ----------------
    elif page == "Admin":
        if not need(role,"users"):
            st.stop()
        st.subheader("Administrator dashboard")
        with db() as c:
            users_df=pd.read_sql_query("SELECT id,username,email,role,locked,created_at,last_login,customer_id FROM users ORDER BY id",c)
            admin_count=c.execute("SELECT COUNT(*) FROM users WHERE role='Admin'").fetchone()[0]
        k1,k2,k3=st.columns(3)
        k1.metric("Total users",len(users_df));k2.metric("Locked users",int(users_df.locked.fillna(0).astype(int).sum()) if not users_df.empty else 0);k3.metric("Admin accounts",int(admin_count))
        if not users_df.empty:
            st.markdown("#### Users")
            show_df(users_df.drop(columns=["id","customer_id"]),400)
            role_counts=users_df.groupby("role").size().rename("users").reset_index()
            st.markdown("#### Users per role"); show_df(role_counts,200)
        st.markdown("#### Add user")
        with st.form("admin_add_user_form"):
            c1,c2=st.columns(2)
            nu=c1.text_input("Username");ne=c2.text_input("Email")
            npw=st.text_input("Temporary password",type="password")
            nr=st.selectbox("Role",["Operations Manager","Freight Broker","Auditor","Customer","Admin"])
            add=st.form_submit_button("Create user",type="primary")
        if add:
            ne=ne.strip().lower();nu=nu.strip();problems=password_problems(npw)
            if not valid_username(nu) or not valid_email(ne): st.error("Enter a valid username and email address.")
            elif problems: st.error(pw_error(problems))
            else:
                with db() as c: dup=c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?) OR lower(email)=lower(?)",(nu,ne)).fetchone()
                if dup: st.error("Username or email already exists.")
                elif password_taken(npw): st.error("This password is already used by another account.")
                else:
                    with db() as c: c.execute("INSERT INTO users(username,email,password_hash,security_question,security_answer_hash,pwd_fp,role,locked,created_at) VALUES(?,?,?,?,?,?,?,?,?)",(nu,ne,hash_txt(npw),SECURITY_QUESTIONS[0],hash_txt("admin-created"),pwd_fp(npw),nr,0,datetime.datetime.now().isoformat(timespec="seconds")))
                    audit_admin(email,"add_user",ne,f"Created {nr} account");st.success("User created.");st.rerun()
        st.markdown("#### Change role / promote / demote")
        editable=users_df[users_df.email!=email] if not users_df.empty else users_df
        if not editable.empty:
            idmap={f"{r.username} · {r.email} ({r.role})":int(r.id) for r in editable.itertuples()}
            chosen=st.selectbox("Select account",list(idmap))
            target_id=idmap[chosen]
            current=users_df[users_df.id==target_id].iloc[0]
            new_role=st.selectbox("New role",["Admin","Operations Manager","Freight Broker","Auditor","Customer"],index=["Admin","Operations Manager","Freight Broker","Auditor","Customer"].index(current.role))
            if st.button("Save role change"):
                if current.role=="Admin" and new_role!="Admin" and admin_count<=1: st.error("The last remaining Admin cannot be demoted.")
                else:
                    with db() as c: c.execute("UPDATE users SET role=? WHERE id=?",(new_role,target_id))
                    audit_admin(email,"change_role",current.email,f"{current.role} -> {new_role}");st.success("Role updated.");st.rerun()
            col1,col2=st.columns(2)
            if col1.button("Lock selected user"):
                if int(current.id)==int(user["id"]): st.error("You cannot lock your own account.")
                elif current.role=="Admin" and admin_count<=1: st.error("The last remaining Admin cannot be locked.")
                else:
                    with db() as c: c.execute("UPDATE users SET locked=1 WHERE id=?",(target_id,))
                    audit_admin(email,"lock_user",current.email,"Account locked");st.success("User locked immediately.");st.rerun()
            if col2.button("Unlock selected user"):
                with db() as c: c.execute("UPDATE users SET locked=0 WHERE id=?",(target_id,))
                audit_admin(email,"unlock_user",current.email,"Account unlocked");st.success("User unlocked.");st.rerun()
            if st.button("Delete selected user",type="secondary"):
                if int(current.id)==int(user["id"]): st.error("You cannot delete your own account.")
                elif current.role=="Admin" and admin_count<=1: st.error("The last remaining Admin cannot be deleted.")
                elif st.session_state.get("confirm_delete_user")==target_id:
                    with db() as c: c.execute("DELETE FROM users WHERE id=?",(target_id,))
                    audit_admin(email,"delete_user",current.email,"Account deleted");st.session_state.confirm_delete_user=None;st.success("User deleted.");st.rerun()
                else:
                    st.session_state.confirm_delete_user=target_id
                    st.warning("Click Delete selected user again to confirm permanent deletion.")
        st.markdown("#### Activity log · latest 50 actions")
        with db() as c: logdf=pd.read_sql_query("SELECT ts,actor,action,target,detail FROM activity_log ORDER BY id DESC LIMIT 50",c)
        show_df(logdf,300)
        st.markdown("#### Role permission matrix")
        st.table(pd.DataFrame(ROLE_MATRIX,columns=["Role","Copilot","Route","Quotes","Carriers","Weather","Margin","Customs","Run agents","Users"]))



## 2. Launch the Streamlit application

This starts **Streamlit only** and opens it through ngrok. Keep this cell running while you use the app.

In [ ]:
import os, time, subprocess
from pyngrok import ngrok, conf

conf.get_default().auth_token = os.environ['NGROK_AUTHTOKEN']
for tunnel in ngrok.get_tunnels():
    try: ngrok.disconnect(tunnel.public_url)
    except Exception: pass
ngrok.kill()
subprocess.run(['bash','-lc','pkill -f "streamlit run app.py" || true'],capture_output=True)
process = subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.address','0.0.0.0'],stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
time.sleep(5)
public_url = ngrok.connect(8501,'http').public_url
print('='*65)
print('🚀 FreightQuote AI — Streamlit URL:',public_url)
print('='*65)
print('The app is running. Stop this cell / Colab runtime to shut it down.')
try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    ngrok.kill(); process.terminate(); print('Streamlit and ngrok stopped.')